GALLAPONT Thibault - MALLE Arnaud - MUEL Thomas

# Projet de NLP : modelling 3, la pipeline « ultime »

Ce notebook garde **seulement ce qui a marché** dans `Modelling2.ipynb` et pousse le levier principal, le **LLM fine-tuné**, beaucoup plus loin. Même protocole (5 folds partagés, choix sur l'OOF, dev seulement pour mesurer), **même cache** (`cache_modelling2/`) : les runs déjà faits (baseline, 5 seeds XLM-R soft, Qwen 7B 4 bits, few-shot) sont relus sans réentraînement.

| Partie | Contenu | Durée estimée (G4, 96 Go) |
|---|---|---|
| 2 | modèle 1 bon marché (TF-IDF, prior mot-clé, prior kNN) : uniquement comme entrées du stacking | ~5 min (cache) |
| 3 | XLM-R large soft : 5 seeds `prep_v2` + multi-tâches, 3 seeds `prep_v2` + 4 classes (diversité) | ~1 h 30 |
| 4 | **LLM 14B en LoRA bf16** (Qwen2.5-14B-Instruct, Qwen3-14B), 3 époques, rang 32 | ~2 h par LLM |
| 5 | XLM-R XL (3,5 B), optionnel | ~1 h |
| 6 | combinaisons : ensemble glouton, mélange pondéré LLM/XLM-R, stacking | quelques min |
| **Total** | | **~6-7 h** (tout est mis en cache après chaque fold) |

## 0. Ce que disent les résultats de modelling 2

| Modèle | OOF F1 (seuil OOF) | dev F1 | Δ vs baseline | p(Δ≤0) |
|---|---|---|---|---|
| baseline (XLM-R large, moyenne hard + soft) | 0,8505 | 0,8678 | — | — |
| XLM-R soft, 3 ép., 2e-5 (1 seed) | 0,8512 | 0,8829 | +1,5 | 0,018 |
| XLM-R soft, moyenne de 5 seeds | 0,8564 | 0,8778 | +1,0 | 0,065 |
| **Qwen2.5-7B LoRA 4 bits** (2 ép.) | **0,8613** | 0,8772 | +0,9 | 0,19 |
| ensemble glouton : LLM + 2 seeds XLM-R | 0,8671 | **0,8882** | +2,0 | **0,007** |
| `stack_tout` (retenu sur l'OOF) | **0,8708** | 0,8825 | +1,5 | 0,051 |

**1. Le LLM fine-tuné est le meilleur modèle seul, et il était sous-entraîné.** Meilleur OOF de tous les modèles simples, avec seulement 7B, quantifié en 4 bits (5,6 Go utilisés sur 96 Go), 2 époques, rang 16. La perte passait de 0,62 à 0,48 entre les époques 1 et 2 et le F1 de chaque fold gagnait encore ~1 point : il n'avait pas fini d'apprendre.

**2. LLM et XLM-R se trompent sur des tweets différents.** Sur le dev : 67 erreurs communes, 45 propres au LLM, 46 propres à XLM-R (corrélation des probabilités 0,94 seulement). C'est pour ça que toutes les meilleures lignes sont des combinaisons LLM + XLM-R. Une simple moyenne 50/50 atteint 0,900 au meilleur seuil du dev (chiffre indicatif, seuil lu sur le dev).

**3. Soft > hard, partout.** Le hard est mal calibré (CE soft 1,27 contre 0,54) et la moyenne hard + soft (la baseline) n'apporte rien par rapport au soft seul. **On abandonne le hard.**

**4. Petits gains réels côté XLM-R** : `prep_v2` (+0,4 pt OOF), multi-tâches (+0,1), moyenne de seeds (+0,5). `cls4` est meilleur sur DIRECT et REPORTED (0,898 et 0,909 contre 0,876 et 0,881) mais moins bon sur NO : faible seul, mais utile pour la diversité de l'ensemble.

**5. Ce qui n'a rien donné** : AEDA, tête CNN, `cls7`, cascade modèle 1, règle lexique, justifications générées (`rationale`), few-shot seul (0,766), stacking HGB, seuil par langue (instable). On les retire.

**6. Deux runs ont divergé sur un fold** (`rationale` et `bs32` : OOF 0,79 mais dev 0,87, signe qu'un des 5 modèles s'est effondré). On ajoute un garde-fou : un fold sous 0,75 de F1 est relancé avec une autre seed.

**7. Les erreurs restantes sont surtout du bruit d'annotation** : 74 des 106 erreurs du modèle retenu sont des tweets à 2 ou 4 votes YES sur 6. Il reste peu de marge sur les tweets non ambigus (exactitude 1,000 à 0 vote, 0,975 à 6 votes).

### Ce que change ce notebook

| Levier | Modelling 2 | Modelling 3 |
|---|---|---|
| Taille du LLM | 7B | **14B** (option 32B) |
| Précision | 4 bits (QLoRA) | **bf16** (pas de perte de quantification) |
| LoRA | rang 16, α 32 | **rang 32, α 64** |
| Époques LLM | 2 (encore en progrès) | **3**, époque choisie sur l'OOF |
| Calendrier | linéaire | **cosinus**, warmup 5 % |
| Consigne | une phrase | **définition EXIST courte** (DIRECT / REPORTED / JUDGEMENTAL + ce qui est NO) |
| Nombre de LLM | 1 | **2 familles** (Qwen2.5, Qwen3) + le 7B déjà en cache : diversité |
| XLM-R | 5 seeds `light` | + 5 seeds `prep_v2` + multi-tâches, + 3 seeds `cls4` |
| Hard / moyenne hard+soft | oui | **non** (baseline seulement, pour la comparaison) |
| Garde-fou de divergence | non | **oui** |
| Combinaison | stacking LR | ensemble glouton, **mélange pondéré LLM / XLM-R**, stacking ; choix sur l'OOF |

## 1. Installation, paramètres et chargement

In [1]:
%pip install -q "transformers>=4.51" "peft>=0.13" accelerate sentencepiece protobuf tqdm scikit-learn scipy

In [2]:
import json, re, html, math, time, gc, random, unicodedata
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM, get_linear_schedule_with_warmup, get_cosine_schedule_with_warmup
from transformers.utils import logging as hf_logging

from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score
from scipy.sparse import hstack

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 220)
hf_logging.set_verbosity_error()

SEED = 42
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert device.type == "cuda", "Ce notebook demande un GPU (Colab : Exécution > Modifier le type d'exécution > G4)"
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(torch.cuda.get_device_name(0), "| précision :", AMP_DTYPE, "| mémoire :",
      round(torch.cuda.get_device_properties(0).total_memory / 1e9), "Go")

NVIDIA RTX PRO 6000 Blackwell Server Edition | précision : torch.bfloat16 | mémoire : 102 Go


**Paramètres d'exécution.** Les résultats déjà en cache ne sont jamais recalculés. Pour gagner du temps : `RUN_XL = False`, ou un seul LLM dans `LLM_RUNS`.

In [3]:
N_FOLDS = 5
RUN_XLMR = True          # partie 3 : XLM-R large
RUN_LLM_FT = True        # partie 4 : LLM fine-tunés en LoRA bf16
RUN_XL = True            # partie 5 : XLM-R XL (3,5 B) ; le moins prioritaire
EMBED_NAME = "intfloat/multilingual-e5-large"
REFERENCE = "baseline"   # la pipeline de départ, pour le test apparié
COLLAPSE_F1 = 0.75       # un fold sous ce F1 a divergé : on le relance avec une autre seed

XLMR_SEEDS = [42, 43, 44, 45, 46]

LLM_COMMON = dict(epochs=3, lr=1e-4, warmup=0.05, r=32, alpha=64, dropout=0.05, bs=16, micro_bs=8, seed=42)
LLM_RUNS = [
    dict(LLM_COMMON, name="llm_qwen25_14b", model="Qwen/Qwen2.5-14B-Instruct"),
    dict(LLM_COMMON, name="llm_qwen3_14b", model="Qwen/Qwen3-14B", epochs=2),
]
# Si le temps le permet, un modèle plus gros (bf16 ≈ 65 Go, tient sur 96 Go avec micro_bs=4) :
# LLM_RUNS.append(dict(LLM_COMMON, name="llm_qwen25_32b", model="Qwen/Qwen2.5-32B-Instruct", lr=5e-5, micro_bs=4, epochs=2))
LLM_TOKEN_BUDGET = 16_000    # tokens par lot en inférence (divisé automatiquement en cas d'OOM)

In [4]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/TweetClassification")
else:
    PROJECT_DIR = Path(".")

DATA_DIR = PROJECT_DIR
CACHE_DIR = PROJECT_DIR / "cache_modelling2"      # même cache que modelling 2 : les runs identiques sont relus
(CACHE_DIR / "runs").mkdir(parents=True, exist_ok=True)
OUTPUT_CSV = PROJECT_DIR / "dev_predictions_v3.csv"
RESULTS_CSV = PROJECT_DIR / "resultats_modelling3.csv"
PREV_PRED_CSV = PROJECT_DIR / "dev_predictions_v2.csv"   # pour comparer au meilleur modèle de modelling 2
PREV_RES_CSV = PROJECT_DIR / "resultats_modelling2.csv"
print("Colab :", IN_COLAB, "| projet :", PROJECT_DIR.resolve(), "| cache :", CACHE_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Colab : True | projet : /content/drive/MyDrive/TweetClassification | cache : /content/drive/MyDrive/TweetClassification/cache_modelling2


In [5]:
TASK2 = ["NO", "DIRECT", "REPORTED", "JUDGEMENTAL"]
TASK3 = ["IDEOLOGICAL-INEQUALITY", "STEREOTYPING-DOMINANCE", "OBJECTIFICATION",
         "SEXUAL-VIOLENCE", "MISOGYNY-NON-SEXUAL-VIOLENCE"]


def load_exist(name):
    path = next(p for p in DATA_DIR.rglob(name) if "cache" not in str(p))
    with open(path, encoding="utf-8") as f:
        df = pd.DataFrame.from_dict(json.load(f), orient="index").reset_index(drop=True)
    df["id"] = df["id_EXIST"].astype(int)
    return df.sort_values("id").reset_index(drop=True)   # l'ordre des id suit les mots-clés de collecte


def task2_dist(row):
    """Répartition des 6 votes entre NO / DIRECT / REPORTED / JUDGEMENTAL."""
    c = np.zeros(4)
    for l1, l2 in zip(row["labels_task1"], row["labels_task2"]):
        c[0 if l1 != "YES" else (TASK2.index(l2) if l2 in TASK2[1:] else 1)] += 1
    return c / c.sum()


def task3_frac(row):
    """Part des annotateurs qui citent chaque catégorie de la tâche 3 (multi-label)."""
    c = np.zeros(5)
    for cats in row["labels_task3"]:
        for cat in (cats if isinstance(cats, list) else [cats]):
            if cat in TASK3:
                c[TASK3.index(cat)] += 1
    return c / len(row["labels_task3"])


def add_labels(df):
    df = df.copy()
    df["n_ann"] = df["labels_task1"].str.len()
    df["n_yes"] = df["labels_task1"].apply(lambda l: sum(x == "YES" for x in l))
    df["p_yes"] = df["n_yes"] / df["n_ann"]
    df["hard"] = np.select([df["p_yes"] > 0.5, df["p_yes"] < 0.5], ["YES", "NO"], default=None)
    df["t2"] = df.apply(task2_dist, axis=1)
    df["t3"] = df.apply(task3_frac, axis=1)
    return df


train_df = add_labels(load_exist("EXIST2024_training.json"))
dev_df = add_labels(load_exist("EXIST2024_dev.json"))
N_TR, N_DEV = len(train_df), len(dev_df)

P_TR, P_DEV = train_df["p_yes"].to_numpy(), dev_df["p_yes"].to_numpy()
TIE_TR, TIE_DEV = train_df["hard"].isna().to_numpy(), dev_df["hard"].isna().to_numpy()
Y_TR = (train_df["hard"] == "YES").astype(int).to_numpy()     # 0 pour les égalités (jamais évaluées)
Y_DEV = (dev_df["hard"] == "YES").astype(int).to_numpy()
NYES_TR = train_df["n_yes"].to_numpy()
T2_TR, T3_TR = np.stack(train_df["t2"]), np.stack(train_df["t3"])
LANG_TR, LANG_DEV = train_df["lang"].to_numpy(), dev_df["lang"].to_numpy()

print(f"Train : {N_TR} tweets ({(~TIE_TR).sum()} avec étiquette hard) | Dev : {N_DEV} ({(~TIE_DEV).sum()})")
print("Répartition moyenne des intentions (train) :", dict(zip(TASK2, T2_TR.mean(0).round(3))))
print("Part moyenne des catégories (train) :", dict(zip(TASK3, T3_TR.mean(0).round(3))))

Train : 6920 tweets (6064 avec étiquette hard) | Dev : 1038 (934)
Répartition moyenne des intentions (train) : {'NO': np.float64(0.548), 'DIRECT': np.float64(0.219), 'REPORTED': np.float64(0.112), 'JUDGEMENTAL': np.float64(0.121)}
Part moyenne des catégories (train) : {'IDEOLOGICAL-INEQUALITY': np.float64(0.14), 'STEREOTYPING-DOMINANCE': np.float64(0.163), 'OBJECTIFICATION': np.float64(0.126), 'SEXUAL-VIOLENCE': np.float64(0.079), 'MISOGYNY-NON-SEXUAL-VIOLENCE': np.float64(0.099)}


## 1 bis. Preprocessing

Identique à modelling 2 : `light` (baseline), `xlmt` (entrée du LLM et des embeddings), `v2` (corrigé, meilleur OOF).

In [6]:
# --- version d'origine (bugs compris) ---
URL_OLD = re.compile(r"https?://\S+|www\.\S+")
MENTION_OLD = re.compile(r"@\w+")
HASHTAG = re.compile(r"#(\w+)")
CAMEL = re.compile(r"(?<=[a-záéíóúñü])(?=[A-ZÁÉÍÓÚÑÜ])")
REPEAT_OLD = re.compile(r"(\w)\1{2,}")


def prep_light(text):
    text = html.unescape(text)
    text = URL_OLD.sub(" url ", text)
    text = MENTION_OLD.sub(" @user ", text)
    text = HASHTAG.sub(lambda m: " " + CAMEL.sub(" ", m.group(1)) + " ", text)
    text = REPEAT_OLD.sub(r"\1\1", text)
    return " ".join(text.split())


# --- version officielle XLM-T, avec des regex correctes ---
TCO = re.compile(r"https?://t\.co/[A-Za-z0-9]{10}")
URL = re.compile(r"https?://\S+|www\.\S+")
MENTION = re.compile(r"(?<![\w@])@\w{1,15}")


def prep_xlmt(text):
    text = html.unescape(text)
    text = TCO.sub(" http ", text)
    text = URL.sub(" http ", text)
    text = MENTION.sub(" @user ", text)
    return " ".join(text.split())


# --- découpage des hashtags tout en minuscules ---
def strip_accents(s):
    return "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))


def build_vocab(texts, min_count=3):
    cnt = Counter()
    for t in texts:
        t = URL.sub(" ", html.unescape(t))
        t = re.sub(r"[@#]\w+", " ", t)
        cnt.update(re.findall(r"[a-z]+", strip_accents(t.lower())))
    extra = ("not all men yes all women me too ni una menos yo si te creo la manada women in tech womens day "
             "violencia machista everyday sexism stop street harassment womens rights we are equal "
             "international feminism feminismo feminista igualdad genero mujer mujeres")
    cnt.update({w: 50 for w in extra.split()})
    return {w: c for w, c in cnt.items() if c >= min_count and (len(w) > 1 or w in "aeoyu")}


VOCAB = build_vocab(train_df["tweet"])
VOCAB_TOTAL = sum(VOCAB.values())


def segment_word(word, max_piece=20):
    """Découpage par programmation dynamique (coût = -log fréquence + 1 par morceau)."""
    w = strip_accents(word.lower())
    n = len(w)
    best = [(0.0, [])] + [(math.inf, None)] * n
    for i in range(1, n + 1):
        for j in range(max(0, i - max_piece), i):
            f = VOCAB.get(w[j:i], 0)
            if f and best[j][0] < math.inf:
                cost = best[j][0] - math.log(f / VOCAB_TOTAL) + 1.0
                if cost < best[i][0]:
                    best[i] = (cost, best[j][1] + [w[j:i]])
    return best[n][1] if best[n][1] else [word]


def split_hashtag(tag):
    if re.search(r"[a-z][A-Z]|[A-Za-z][0-9]", tag):                 # CamelCase ou lettres+chiffres
        return re.sub(r"(?<=[a-záéíóúñü])(?=[A-ZÁÉÍÓÚÑÜ])|(?<=[A-Za-z])(?=[0-9])", " ", tag).split()
    if len(tag) >= 6 and (tag.islower() or tag.isupper()) and VOCAB.get(strip_accents(tag.lower()), 0) < 5:
        pieces = segment_word(tag)
        if len(pieces) > 1:
            return pieces
    return [tag]


REPEAT = re.compile(r"([^\W\d_])\1{2,}")
LAUGH = re.compile(r"\b([jh])([aeiou])(?:\1\2)+[jh]?\b|\b(?:js){2,}j?\b", re.I)   # jajaja, hahaha, jsjsjs (pas « hijo »)
GLUED = re.compile(r"(?<=[a-záéíóúñü])([.!?…])(?=[A-ZÁÉÍÓÚÑ¿¡])")


def prep_v2(text):
    text = html.unescape(text)
    text = re.sub(r"^RT\s+@?\w+:\s*", "", text)
    text = TCO.sub(" http ", text)
    text = URL.sub(" http ", text)
    text = MENTION.sub(" @user ", text)
    text = re.sub(r"(?:@user\s*){2,}", "@user ", text)
    text = re.sub(r"(?:http\s*){2,}", "http ", text)
    text = HASHTAG.sub(lambda m: " #" + " ".join(split_hashtag(m.group(1))) + " ", text)
    text = LAUGH.sub(lambda m: m.group(0)[:2] * 2, text)
    text = REPEAT.sub(r"\1\1", text)
    text = GLUED.sub(r"\1 ", text)
    return " ".join(text.split())


PREPS = {"light": prep_light, "xlmt": prep_xlmt, "v2": prep_v2}
TEXTS = {name: (train_df["tweet"].map(f).to_numpy(), dev_df["tweet"].map(f).to_numpy())
         for name, f in PREPS.items()}

examples = [s for s in train_df["tweet"] if re.search(r"#[a-z]{8,}\b|t\.co/\w{10}[A-Za-z]|\w@\w|\d{4,}|@\w+\s+@\w+\s+@\w+", s)][:8]
display(pd.DataFrame({name: [f(s) for s in examples] for name, f in PREPS.items()}, index=[s[:60] for s in examples]))
print([(h, split_hashtag(h)) for h in ["notallmen", "yositecreo", "violenciamachista", "lamanada", "womenintech", "IDoNotRenounce"]])

,light,xlmt,v2
"@Steven2897 Lee sobre Gamergate, y como eso ha cambiado la m","@user Lee sobre Gamergate, y como eso ha cambiado la manera en la cual nos comunicamos en el internet. Los fanboys de Halo están tóxicos...","@user Lee sobre Gamergate, y como eso ha cambiado la manera en la cual nos comunicamos en el internet. Los fanboys de Halo están tóxicos...","@user Lee sobre Gamergate, y como eso ha cambiado la manera en la cual nos comunicamos en el internet. Los fanboys de Halo están tóxicos..."
@novadragon21 @icep4ck @TvDannyZ Entonces como así es el mer,@user @user @user Entonces como así es el mercado lo mejor no es hacer algo para cambiarlo y seguir alimentando el machismo en los consu...,@user @user @user Entonces como así es el mercado lo mejor no es hacer algo para cambiarlo y seguir alimentando el machismo en los consu...,@user Entonces como así es el mercado lo mejor no es hacer algo para cambiarlo y seguir alimentando el machismo en los consumidores en l...
.¿Conoces la #DECORACION #estilo #GAMER para #adolescentes,.¿Conoces la DECORACION estilo GAMER para adolescentes ?Te lo explicamos en nuestro post : ❤️ url sigues? inspiracionalcuadrado interior...,.¿Conoces la #DECORACION #estilo #GAMER para #adolescentes ?Te lo explicamos en nuestro #post : ❤️ http ¿Nos sigues?#inspiracionalcuadra...,.¿Conoces la #DECORACION #estilo #GAMER para #adolescentes ?Te lo explicamos en nuestro #post : ❤️ http ¿Nos sigues? #inspiracionalcuadr...
CES 2022 ASUS ROG Rise of Gamers Evento de lanzamiento https,CES 2022 ASUS ROG Rise of Gamers Evento de lanzamiento url AMD2022 AMD AMDRed Team AMDRyzen Intel nvidia NVIDIAOmniverse NVIDIAGe Force ...,CES 2022 ASUS ROG Rise of Gamers Evento de lanzamiento http #AMD2022 #AMD #AMDRedTeam #AMDRyzen #Intel #nvidia #NVIDIAOmniverse #NVIDIAG...,CES 2022 ASUS ROG Rise of Gamers Evento de lanzamiento http #AMD 2022 #AMD #AMDRed Team #AMDRyzen #Intel #nvidia #NVIDIAOmniverse #NVIDI...
¿Y por qué digo que son dogwhistles? Porque esto ya lo hemos,"¿Y por qué digo que son dogwhistles? Porque esto ya lo hemos vivido una y otra y otra vez en los cómics... Pasó con Comics Gate en 2017,...","¿Y por qué digo que son dogwhistles? Porque esto ya lo hemos vivido una y otra y otra vez en los cómics... Pasó con #ComicsGate en 2017,...",¿Y por qué digo que son dogwhistles? Porque esto ya lo hemos vivido una y otra y otra vez en los cómics... Pasó con #Comics Gate en 2017...
La que se levanto sin miedo al exito fue la rata Nro 1 de Ru,La que se levanto sin miedo al exito fue la rata Nro 1 de Rust.... @user url gamergirls GAMERGATE url,La que se levanto sin miedo al exito fue la rata Nro 1 de Rust.... @user http #Rust #gamergirls #GAMERGATE http,La que se levanto sin miedo al exito fue la rata Nro 1 de Rust.... @user http #Rust #gamer girls #GAMERGATE http
@BrunoLouviers en 2014 empezó gamergate así que igual mejor,@user en 2014 empezó gamergate así que igual mejor no,@user en 2014 empezó gamergate así que igual mejor no,@user en 2014 empezó gamergate así que igual mejor no
Los mejores 100 #videojuegos de la #Historia 😀😀😀😀 #gamers #G,Los mejores 100 videojuegos de la Historia 😀😀😀😀 gamers Gamer Gate url,Los mejores 100 #videojuegos de la #Historia 😀😀😀😀 #gamers #GamerGate http,Los mejores 100 #video juegos de la #Historia 😀😀😀😀 #gamers #Gamer Gate http


[('notallmen', ['notallmen']), ('yositecreo', ['yo', 'site', 'creo']), ('violenciamachista', ['violencia', 'machista']), ('lamanada', ['lamanada']), ('womenintech', ['women', 'in', 'tech']), ('IDoNotRenounce', ['IDo', 'Not', 'Renounce'])]


## 1 ter. Protocole d'évaluation

Identique : 5 folds stratifiés (langue × votes), seed 42, seuil choisi sur l'OOF, bootstrap apparié sur le dev.

In [7]:
strat = train_df["lang"] + "_" + train_df["n_yes"].astype(str)
FOLDS = np.zeros(N_TR, dtype=int)
for k, (_, va) in enumerate(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(np.zeros(N_TR), strat)):
    FOLDS[va] = k

EVAL_TR, EVAL_DEV = ~TIE_TR, ~TIE_DEV
THR_GRID = np.round(np.arange(0.20, 0.801, 0.01), 2)
REG = {}          # nom -> {"oof", "dev", "kind", ...}


def clip(p):
    return np.clip(np.asarray(p, dtype=float), 1e-4, 1 - 1e-4)


def logit(p):
    p = clip(p)
    return np.log(p / (1 - p))


def f1_at(y, p, thr=0.5):
    return f1_score(y, (p >= thr).astype(int), zero_division=0)


def best_threshold(p_oof):
    m = EVAL_TR & ~np.isnan(p_oof)
    scores = [f1_at(Y_TR[m], p_oof[m], t) for t in THR_GRID]
    i = int(np.argmax(scores))
    return float(THR_GRID[i]), scores[i]


def soft_ce(p_true, p_pred):
    m = ~np.isnan(p_pred)
    p = clip(p_pred[m])
    return float(-np.mean(p_true[m] * np.log(p) + (1 - p_true[m]) * np.log(1 - p)))


RNG_BOOT = np.random.default_rng(0)
BOOT_IDX = RNG_BOOT.integers(0, EVAL_DEV.sum(), size=(2000, EVAL_DEV.sum()))


def boot_f1(pred):
    """F1 vectorisé sur 2 000 rééchantillonnages du dev (mêmes tirages pour tous les modèles)."""
    y = Y_DEV[EVAL_DEV][BOOT_IDX]
    p = pred[BOOT_IDX]
    tp = (y * p).sum(1); fp = ((1 - y) * p).sum(1); fn = (y * (1 - p)).sum(1)
    return 2 * tp / np.maximum(2 * tp + fp + fn, 1)


def register(name, oof, dev, kind, **info):
    REG[name] = dict(oof=np.asarray(oof, dtype=float), dev=np.asarray(dev, dtype=float), kind=kind, **info)


def oof_score(name):
    return best_threshold(REG[name]["oof"])[1]


def evaluate(name, ref=None):
    ref = ref or REFERENCE
    r = REG[name]
    thr, f1_oof_best = best_threshold(r["oof"])
    pred_dev = (r["dev"][EVAL_DEV] >= thr).astype(int)
    bs = boot_f1(pred_dev)
    row = {
        "nom": name, "type": r["kind"],
        "OOF F1@0.5": f1_at(Y_TR[EVAL_TR & ~np.isnan(r["oof"])], r["oof"][EVAL_TR & ~np.isnan(r["oof"])]),
        "seuil OOF": thr, "OOF F1@seuil": f1_oof_best,
        "OOF CE soft": soft_ce(P_TR, r["oof"]),
        "dev F1@0.5": f1_at(Y_DEV[EVAL_DEV], r["dev"][EVAL_DEV]),
        "dev F1@seuil": f1_at(Y_DEV[EVAL_DEV], r["dev"][EVAL_DEV], thr),
        "IC95 bas": np.percentile(bs, 2.5), "IC95 haut": np.percentile(bs, 97.5),
        "dev EN": f1_at(Y_DEV[EVAL_DEV & (LANG_DEV == "en")], r["dev"][EVAL_DEV & (LANG_DEV == "en")], thr),
        "dev ES": f1_at(Y_DEV[EVAL_DEV & (LANG_DEV == "es")], r["dev"][EVAL_DEV & (LANG_DEV == "es")], thr),
        "dev CE soft": soft_ce(P_DEV, r["dev"]),
    }
    if ref in REG and ref != name:
        thr_ref, _ = best_threshold(REG[ref]["oof"])
        bs_ref = boot_f1((REG[ref]["dev"][EVAL_DEV] >= thr_ref).astype(int))
        row["Δ dev vs réf"] = row["dev F1@seuil"] - f1_at(Y_DEV[EVAL_DEV], REG[ref]["dev"][EVAL_DEV], thr_ref)
        row["p(Δ≤0)"] = float(np.mean(bs - bs_ref <= 0))
    return row


def summary(names=None, sort="OOF F1@seuil", ref=None):
    names = list(dict.fromkeys(names or REG))
    df = pd.DataFrame([evaluate(n, ref) for n in names]).set_index("nom")
    return df.sort_values(sort, ascending=False).round(4)


def compare_dev(pred_new, pred_ref):
    """Écart de F1 sur le dev et part des rééchantillonnages bootstrap où il est ≤ 0 (prédictions 0/1 sur EVAL_DEV)."""
    d = boot_f1(pred_new) - boot_f1(pred_ref)
    yv = Y_DEV[EVAL_DEV]
    return f1_score(yv, pred_new) - f1_score(yv, pred_ref), float(np.mean(d <= 0))

## 2. Modèle 1 bon marché (entrées du stacking)

TF-IDF, features manuelles, prior du mot-clé, prior kNN : seuls ils sont faibles, mais `stack_tout` (+0,2 pt OOF sur `stack_bert+llm`) montre qu'ils apportent un peu au stacking. Ils ne coûtent rien (cache).

In [8]:
def norm_text(t):
    """Minuscules, sans accents, hashtags découpés, ponctuation retirée : pour TF-IDF et les lexiques."""
    t = html.unescape(t)
    t = TCO.sub(" http ", t); t = URL.sub(" http ", t)
    t = MENTION.sub(" atuser ", t)
    t = HASHTAG.sub(lambda m: " " + " ".join(split_hashtag(m.group(1))) + " " + m.group(1) + " ", t)
    t = strip_accents(t.lower())
    t = REPEAT.sub(r"\1\1", t)
    return " ".join(re.sub(r"[^a-z0-9ñ ]+", " ", t).split())


NORM_TR = train_df["tweet"].map(norm_text).to_numpy()
NORM_DEV = dev_df["tweet"].map(norm_text).to_numpy()


def wb(words):
    return re.compile(r"\b(?:" + "|".join(words) + r")\b")


LEXICONS = {   # appliqués au texte normalisé
    "lex_sexist": wb([r"feminazis?", r"whores?", r"sluts?", r"slutbag\w*", r"skanks?", r"cunts?", r"zorras?", r"guarras?",
                      r"golfas?", r"marimachos?", r"sumisas?", r"dumb blondes?", r"rubias? tontas?", r"hembrismo", r"hembrista\w*",
                      r"mujer florero", r"gold diggers?", r"make me a sandwich", r"frigidas?", r"lagartonas?", r"nenazas?",
                      r"machorras?", r"furcias?", r"fulanas?", r"rameras?", r"thots?", r"femoids?", r"foids?"]),
    "generalisation": re.compile(r"\b(?:women|woman|girls|females|ladies|wives|mujeres|una mujer|las chicas|las ninas|"
                                 r"las senoritas|feministas|feminists)\s+(?:are|r|can t|cannot|don t|do not|shouldn t|should|"
                                 r"always|never|need|just|only|belong|son|estan|no saben|no pueden|no deben|no deberian|no sirven|"
                                 r"siempre|nunca|solo|deben|deberian|necesitan)\b"),
    "trad_role": wb([r"kitchen", r"sandwich\w*", r"cook for", r"dishes", r"laundry", r"housewife", r"cocina\w*", r"fregar",
                     r"fregona", r"lavar (?:los platos|la ropa)", r"ama de casa", r"a la casa", r"planchar"]),
    "feminism_vocab": wb([r"feminis\w*", r"feminazi\w*", r"hembris\w*", r"sororidad", r"patriarca\w*", r"patriarchy",
                          r"patriarchal", r"8m", r"ni una menos", r"niunamenos", r"empoderad\w*"]),
    "sexism_vocab": wb([r"sexis\w*", r"misogin\w*", r"misogyn\w*", r"machis\w*", r"micromachis\w*", r"mansplain\w*",
                        r"manspread\w*", r"acos\w*", r"harass\w*", r"violaci\w*", r"violad\w*", r"rape\w*", r"rapist\w*",
                        r"abus\w*", r"maltrat\w*", r"brecha", r"glass ceiling", r"techo de cristal", r"desigualdad",
                        r"inequality", r"gender gap", r"pay gap", r"feminicid\w*", r"femicid\w*", r"violencia de genero",
                        r"violencia machista", r"cosifica\w*", r"objectif\w*", r"consent\w*", r"discrimina\w*"]),
    "gender_insult": wb([r"bitch\w*", r"whore\w*", r"slut\w*", r"cunt\w*", r"hoes?", r"thots?", r"skank\w*", r"putas?",
                         r"zorras?", r"perras?", r"guarras?", r"golfas?", r"furcias?", r"pendeja\w*", r"tontas?", r"gordas?",
                         r"feas?"]),
    "women_ref": wb([r"women", r"woman", r"girls?", r"females?", r"she", r"her", r"hers", r"lady", r"ladies", r"wife",
                     r"wives", r"mom", r"mother", r"mujer\w*", r"chicas?", r"ninas?", r"senoras?", r"senoritas?",
                     r"esposas?", r"novias?", r"ellas?", r"madres?", r"morras?", r"minas?"]),
    "men_ref": wb([r"men", r"man", r"guys?", r"males?", r"boys?", r"he", r"him", r"his", r"husband", r"hombres?",
                   r"chicos?", r"tios?", r"varon\w*", r"machos?", r"maridos?", r"novios?", r"ellos"]),
    "body_sex": wb([r"tits", r"boobs", r"ass", r"pussy", r"dick", r"cock", r"cum", r"sexy", r"nudes?", r"culo\w*",
                    r"tetas?", r"chichis", r"nalgas", r"follar\w*", r"coger", r"pollas?", r"vergas?", r"vagina\w*",
                    r"panocha", r"cuerpo\w*", r"body", r"bikini", r"faldas?", r"skirts?", r"sujetador", r"bragas"]),
    "personal_exp": re.compile(r"\b(?:me (?:dijo|dijeron|grito|gritaron|toco|tocaron|siguio|siguieron|acoso|acosaron|"
                               r"llamo|llamaron)|i (?:was|got|ve been|have been) (?:harassed|raped|assaulted|catcalled|groped|"
                               r"followed|called)|sufri\w*|my (?:boss|ex|husband|boyfriend)|mi (?:jefe|ex|marido|novio))\b"),
    "negation": wb([r"not", r"no", r"never", r"nunca", r"jamas", r"ni", r"don t", r"doesn t", r"isn t"]),
    "second_person": wb([r"you", r"your", r"u", r"ur", r"tu", r"te", r"vos", r"usted", r"ustedes", r"tus", r"ti"]),
    "first_person": wb([r"i", r"me", r"my", r"yo", r"mi", r"mis"]),
    "laugh_word": wb([r"jaja\w*", r"haha\w*", r"lol", r"lmao", r"jsjs\w*", r"xd+"]),
}
RAW_PATTERNS = {   # appliqués au texte brut
    "denunciation_hashtag": re.compile(r"#(?:metoo|yositecreo|niunamenos|8m|cuentalo|everydaysexism|stopstreetharassment|"
                                       r"violenciamachista|timesup|yesallwomen|notallmen)\b", re.I),
    "campaign_hashtag": re.compile(r"#(?:womensday|internationalwomensday|iwd\w*|diadelamujer\w*|diainternacionaldelamujer|"
                                   r"womenintech|womenempowerment|weareequal|womensrights|empoderamiento\w*|"
                                   r"womensupportingwomen|girlpower)\b", re.I),
    "has_hashtag": re.compile(r"#\w+"),
    "has_url": re.compile(r"https?://"),
    "n_mentions": re.compile(r"(?<![\w@])@\w+"),
    "question": re.compile(r"\?"),
    "quotes": re.compile(r"[\"“”«»]"),
    "laugh_emoji": re.compile(r"[😂🤣]"),
    "exasperation_emoji": re.compile(r"[🙄🤦🤷😒😤]"),
    "heart_emoji": re.compile(r"[❤💕💖💗💜💚💙🥰😍😘]"),
}
FEATURE_NAMES = list(LEXICONS) + list(RAW_PATTERNS) + ["is_reply", "log_len"]


def hand_features(raw, normed):
    rows = []
    for t, n in zip(raw, normed):
        r = [min(len(p.findall(n)), 3) for p in LEXICONS.values()]
        r += [min(len(p.findall(t)), 3) for p in RAW_PATTERNS.values()]
        r += [float(t.lstrip().startswith("@")), math.log1p(len(n.split()))]
        rows.append(r)
    return np.array(rows, dtype=np.float32)


HF_TR = hand_features(train_df["tweet"], NORM_TR)
HF_DEV = hand_features(dev_df["tweet"], NORM_DEV)
_mu, _sd = HF_TR.mean(0), HF_TR.std(0) + 1e-6
HFZ_TR, HFZ_DEV = (HF_TR - _mu) / _sd, (HF_DEV - _mu) / _sd   # centrées-réduites (stats du train)

# Part de YES quand la feature est présente (train, hors égalités) : vérifie que chaque feature porte un signal
stats = []
for j, name in enumerate(FEATURE_NAMES[:-1]):
    present = (HF_TR[:, j] > 0) & EVAL_TR
    stats.append({"feature": name, "part du train": present.mean(), "part de YES si présente": Y_TR[present].mean() if present.any() else np.nan})
display(pd.DataFrame(stats).set_index("feature").round(3).sort_values("part de YES si présente", ascending=False))

,part du train,part de YES si présente
feature,,
generalisation,0.042,0.873
lex_sexist,0.061,0.870
trad_role,0.005,0.816
gender_insult,0.063,0.747
feminism_vocab,0.064,0.733
women_ref,0.301,0.712
body_sex,0.062,0.641
sexism_vocab,0.141,0.631
exasperation_emoji,0.016,0.615


In [9]:
def soft_logreg_fit(X, p, C):
    """Régression logistique sur cibles soft : chaque tweet dupliqué (YES poids p, NO poids 1-p)."""
    from scipy.sparse import vstack, issparse
    Xd = vstack([X, X]) if issparse(X) else np.vstack([X, X])
    yd = np.r_[np.ones(X.shape[0]), np.zeros(X.shape[0])]
    w = np.r_[p, 1 - p]
    return LogisticRegression(C=C, max_iter=3000).fit(Xd, yd, sample_weight=w)


def oof_and_dev(make_X, C):
    """make_X(train_idx) -> (X_train, transform) ; renvoie les probabilités OOF et dev (modèle refit sur tout le train)."""
    oof = np.zeros(N_TR)
    for k in range(N_FOLDS):
        tr, va = FOLDS != k, FOLDS == k
        Xtr, transform = make_X(tr)
        oof[va] = soft_logreg_fit(Xtr, P_TR[tr], C).predict_proba(transform("tr", va))[:, 1]
    Xall, transform = make_X(np.ones(N_TR, bool))
    dev = soft_logreg_fit(Xall, P_TR, C).predict_proba(transform("dev", None))[:, 1]
    return oof, dev


def make_tfidf(mask):
    w = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True).fit(NORM_TR[mask])
    c = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=3, sublinear_tf=True, max_features=200_000).fit(NORM_TR[mask])
    tf = lambda texts: hstack([w.transform(texts), c.transform(texts)]).tocsr()
    return tf(NORM_TR[mask]), lambda split, idx: tf(NORM_TR[idx] if split == "tr" else NORM_DEV)


oof, dev = oof_and_dev(make_tfidf, C=2.0)
register("tfidf", oof, dev, "modèle 1")



def make_hand(mask):
    return HFZ_TR[mask], lambda split, idx: HFZ_TR[idx] if split == "tr" else HFZ_DEV


oof, dev = oof_and_dev(make_hand, C=1.0)
register("features_manuelles", oof, dev, "modèle 1")
summary(["tfidf", "features_manuelles"])

,type,OOF F1@0.5,seuil OOF,OOF F1@seuil,OOF CE soft,dev F1@0.5,dev F1@seuil,IC95 bas,IC95 haut,dev EN,dev ES,dev CE soft
nom,,,,,,,,,,,,
tfidf,modèle 1,0.7468,0.44,0.7743,0.5777,0.7665,0.7894,0.7582,0.8171,0.7766,0.7985,0.5716
features_manuelles,modèle 1,0.6809,0.38,0.7549,0.6105,0.7217,0.7825,0.7520,0.8088,0.7650,0.7964,0.6026


In [10]:
def ngram_set(normed, nmax=4):
    toks = normed.split()
    return {" ".join(toks[i:i + n]) for n in range(1, nmax + 1) for i in range(len(toks) - n + 1)}


NG_TR = [ngram_set(t) for t in NORM_TR]
NG_DEV = [ngram_set(t) for t in NORM_DEV]
DOC_FREQ = Counter(g for s in NG_TR for g in s)

seed_cache = CACHE_DIR / "seed_keywords.json"
if seed_cache.exists():
    block_seed = json.load(open(seed_cache, encoding="utf-8"))
else:
    W = 12
    block_seed = []
    for i in tqdm(range(N_TR), desc="mots-clés"):
        win = [j for j in range(max(0, i - W), min(N_TR, i + W + 1)) if j != i and LANG_TR[j] == LANG_TR[i]]
        best, best_score = None, 0.0
        for g in NG_TR[i]:
            if len(g) < 3 or DOC_FREQ[g] > 400 or g in {"atuser", "http"}:
                continue
            c = sum(g in NG_TR[j] for j in win)
            score = c + 0.3 * len(g.split())
            if c >= 4 and score > best_score:
                best, best_score = g, score
        block_seed.append(best)
    json.dump(block_seed, open(seed_cache, "w", encoding="utf-8"))

seed_counts = Counter(s for s in block_seed if s)
SEED_LEX = {s: c for s, c in seed_counts.items() if c >= 5}


def match_seed(ngrams):
    cands = [g for g in ngrams if g in SEED_LEX]
    return max(cands, key=lambda g: (len(g.split()), SEED_LEX[g])) if cands else None


SEED_TR = np.array([match_seed(s) for s in NG_TR], dtype=object)
SEED_DEV = np.array([match_seed(s) for s in NG_DEV], dtype=object)
print(f"Lexique : {len(SEED_LEX)} mots-clés | couverture train {np.mean(SEED_TR != None):.1%}, dev {np.mean(SEED_DEV != None):.1%}")


def target_encode(keys_fit, p_fit, keys_apply, k=10):
    prior = p_fit.mean()
    s = pd.DataFrame({"key": keys_fit, "p": p_fit}).dropna().groupby("key")["p"].agg(["sum", "count"])
    enc = ((s["sum"] + k * prior) / (s["count"] + k)).to_dict()
    return np.array([enc.get(key, prior) for key in keys_apply])


oof = np.zeros(N_TR)
for k in range(N_FOLDS):
    tr, va = FOLDS != k, FOLDS == k
    oof[va] = target_encode(SEED_TR[tr], P_TR[tr], SEED_TR[va])
register("prior_mot_cle", oof, target_encode(SEED_TR, P_TR, SEED_DEV), "modèle 1")

kw = pd.DataFrame({"mot-clé": SEED_TR, "p": P_TR}).dropna().groupby("mot-clé")["p"].agg(["mean", "count"])
kw = kw[kw["count"] >= 10].sort_values("mean")
print("Mots-clés les moins sexistes :", list(kw.index[:12]))
print("Mots-clés les plus sexistes :", list(kw.index[-12:]))
print("Corrélation prior / part de YES — OOF :", np.corrcoef(oof, P_TR)[0, 1].round(3),
      "| dev :", np.corrcoef(REG["prior_mot_cle"]["dev"], P_DEV)[0, 1].round(3))
summary(["tfidf", "features_manuelles", "prior_mot_cle"])

Lexique : 355 mots-clés | couverture train 98.0%, dev 97.2%
Mots-clés les moins sexistes : ['estrecha', 'conciliar', 'doble jornada', 'ecofeminismo', 'pnd', 'corresponsabilidad', 'canonization', 'marginalize', 'repression', 'metanarrative', 'essentialism', 'deserve']
Mots-clés les plus sexistes : ['bien como un hombre', 'feminazi', 'cosificacion del cuerpo femenino', 'rubia tonta', 'marimacho', 'mujer al volante', 'entender a las mujeres', 'las mujeres no', 'mucho feminismo pero', 'de hoy en dia', 'dumb blonde', 'i hate women']
Corrélation prior / part de YES — OOF : 0.592 | dev : 0.579


,type,OOF F1@0.5,seuil OOF,OOF F1@seuil,OOF CE soft,dev F1@0.5,dev F1@seuil,IC95 bas,IC95 haut,dev EN,dev ES,dev CE soft
nom,,,,,,,,,,,,
tfidf,modèle 1,0.7468,0.44,0.7743,0.5777,0.7665,0.7894,0.7582,0.8171,0.7766,0.7985,0.5716
features_manuelles,modèle 1,0.6809,0.38,0.7549,0.6105,0.7217,0.7825,0.7520,0.8088,0.7650,0.7964,0.6026
prior_mot_cle,modèle 1,0.7073,0.46,0.7363,0.6105,0.7036,0.7325,0.6990,0.7635,0.7212,0.7409,0.6105


In [11]:
def embed_texts(texts, name=EMBED_NAME, bs=128):
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModel.from_pretrained(name, torch_dtype=torch.float16).to(device).eval()
    order = np.argsort([len(t) for t in texts])
    out = np.zeros((len(texts), model.config.hidden_size), dtype=np.float32)
    with torch.no_grad():
        for s in tqdm(range(0, len(texts), bs), desc="embeddings"):
            idx = order[s:s + bs]
            enc = tok(["query: " + texts[i] for i in idx], truncation=True, max_length=256, padding=True, return_tensors="pt").to(device)
            h = model(**enc).last_hidden_state
            m = enc["attention_mask"].unsqueeze(-1).to(h.dtype)
            e = (h * m).sum(1) / m.sum(1)
            out[idx] = F.normalize(e.float(), dim=-1).cpu().numpy()
    del model; gc.collect(); torch.cuda.empty_cache()
    return out


emb_cache = CACHE_DIR / "embeddings_e5.npz"
if emb_cache.exists():
    z = np.load(emb_cache); EMB_TR, EMB_DEV = z["tr"], z["dev"]
else:
    EMB_TR = embed_texts(list(TEXTS["xlmt"][0])); EMB_DEV = embed_texts(list(TEXTS["xlmt"][1]))
    np.savez(emb_cache, tr=EMB_TR, dev=EMB_DEV)

SIM_TR = EMB_TR @ EMB_TR.T          # 6 920 × 6 920
SIM_DEV = EMB_DEV @ EMB_TR.T        # 1 038 × 6 920


def allowed_neighbours(i_query=None):
    """Masque des tweets du train utilisables comme voisins (autres folds, pas de quasi-doublon)."""
    if i_query is None:
        return np.ones(N_TR, bool)
    return (FOLDS != FOLDS[i_query]) & (SIM_TR[i_query] < 0.98)


def knn_prior(sims, allowed, k=25, temp=0.05):
    s = np.where(allowed, sims, -np.inf)
    top = np.argpartition(-s, k)[:k]
    w = np.exp((s[top] - s[top].max()) / temp)
    return float((w * P_TR[top]).sum() / w.sum())


oof = np.array([knn_prior(SIM_TR[i], allowed_neighbours(i)) for i in range(N_TR)])
dev = np.array([knn_prior(SIM_DEV[i], allowed_neighbours()) for i in range(N_DEV)])
register("prior_knn", oof, dev, "modèle 1")
summary(["tfidf", "features_manuelles", "prior_mot_cle", "prior_knn"])

,type,OOF F1@0.5,seuil OOF,OOF F1@seuil,OOF CE soft,dev F1@0.5,dev F1@seuil,IC95 bas,IC95 haut,dev EN,dev ES,dev CE soft
nom,,,,,,,,,,,,
prior_knn,modèle 1,0.7895,0.57,0.8035,0.5967,0.8176,0.8107,0.7818,0.8370,0.7737,0.8378,0.5742
tfidf,modèle 1,0.7468,0.44,0.7743,0.5777,0.7665,0.7894,0.7582,0.8171,0.7766,0.7985,0.5716
features_manuelles,modèle 1,0.6809,0.38,0.7549,0.6105,0.7217,0.7825,0.7520,0.8088,0.7650,0.7964,0.6026
prior_mot_cle,modèle 1,0.7073,0.46,0.7363,0.6105,0.7036,0.7325,0.6990,0.7635,0.7212,0.7409,0.6105


In [12]:
def stack_matrix(names, with_hand=False, with_lang=True):
    cols_tr = [logit(REG[n]["oof"]) for n in names]
    cols_dev = [logit(REG[n]["dev"]) for n in names]
    if with_lang:
        cols_tr.append((LANG_TR == "en").astype(float)); cols_dev.append((LANG_DEV == "en").astype(float))
    Xtr, Xdev = np.column_stack(cols_tr), np.column_stack(cols_dev)
    if with_hand:
        Xtr, Xdev = np.hstack([Xtr, HFZ_TR]), np.hstack([Xdev, HFZ_DEV])
    return Xtr, Xdev


def fit_stack(name, names, with_hand=False, model="lr", C=0.5):
    Xtr, Xdev = stack_matrix(names, with_hand)
    make = (lambda: make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=3000))) if model == "lr" else \
           (lambda: HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_leaf_nodes=15, l2_regularization=1.0, random_state=SEED))
    m = EVAL_TR
    oof = np.full(N_TR, np.nan)
    for k in range(N_FOLDS):
        tr, va = m & (FOLDS != k), FOLDS == k
        oof[va] = make().fit(Xtr[tr], Y_TR[tr]).predict_proba(Xtr[va])[:, 1]
    dev = make().fit(Xtr[m], Y_TR[m]).predict_proba(Xdev)[:, 1]
    register(name, oof, dev, "combinaison", inputs=names, hand=with_hand, meta=model)


MODEL1_INPUTS = ["tfidf", "prior_mot_cle", "prior_knn"]

## 3. XLM-R large (soft uniquement)

Cadre commun repris tel quel (`TweetClassifier`, cibles soft / multi-tâches / 4 classes). Deux changements :

- **garde-fou de divergence** : si un fold finit sous `COLLAPSE_F1` à toutes les époques, il est relancé avec une autre seed (fichier `fold{k}_retry{n}.npz`) ;
- **moyenne de seeds** : chaque configuration est entraînée avec plusieurs seeds, puis l'époque est choisie sur l'OOF de la moyenne.

Configurations :

| Nom | Réglages | Pourquoi |
|---|---|---|
| `base_soft`, `base_hard`, `baseline` | modelling 2 (cache) | référence du test apparié |
| `xlmr_light` | soft, 3 ép., 2e-5, `light`, 5 seeds (cache) | le meilleur XLM-R de modelling 2 |
| `xlmr_v2_mt` | + `prep_v2` + multi-tâches, 5 seeds | les deux seuls gains d'ablation, cumulés |
| `xlmr_v2_cls4` | `prep_v2` + 4 classes, 3 seeds | meilleur sur DIRECT / REPORTED : diversité |

In [13]:
TOKENIZERS = {}


def get_tokenizer(name):
    if name not in TOKENIZERS:
        TOKENIZERS[name] = AutoTokenizer.from_pretrained(name)
    return TOKENIZERS[name]


N_OUT = {"hard": 2, "soft": 2, "multitask": 2, "distill": 2, "cls4": 4, "cls7": 7}
SOFT_TARGETS = {"soft": P_TR, "multitask": P_TR}   # "distill" est ajouté en partie 9.3


class TweetClassifier(nn.Module):
    def __init__(self, model_name, head="cls", target="soft", n_feats=0, dropout=0.1,
                 cnn_filters=256, cnn_widths=(4, 6, 8)):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        h = self.encoder.config.hidden_size
        self.head = head
        self.drop = nn.Dropout(dropout)
        if head == "cnn":
            self.convs = nn.ModuleList([nn.Conv1d(h, cnn_filters, k, padding=k // 2) for k in cnn_widths])
            self.proj = nn.Sequential(nn.Linear(cnn_filters * len(cnn_widths), 256), nn.ReLU())
            d = 256
        else:
            self.proj = nn.Sequential(nn.Linear(h, h), nn.Tanh())
            d = h
        n_out = N_OUT[target]
        self.out = nn.Linear(d, n_out)
        self.aux2 = nn.Linear(d, 4) if target == "multitask" else None
        self.aux3 = nn.Linear(d, 5) if target == "multitask" else None
        self.feat = nn.Linear(n_feats, n_out) if n_feats else None
        if self.feat is not None:
            nn.init.zeros_(self.feat.weight); nn.init.zeros_(self.feat.bias)

    def forward(self, input_ids, attention_mask, feats=None):
        hs = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        if self.head == "cls":
            z = hs[:, 0]
        elif self.head == "mean":
            m = attention_mask.unsqueeze(-1).to(hs.dtype)
            z = (hs * m).sum(1) / m.sum(1).clamp(min=1)
        else:
            x = hs.transpose(1, 2)
            L = x.size(2)
            pooled = []
            for conv in self.convs:
                c = F.relu(conv(x))[:, :, :L]                       # largeur paire : on recoupe à L
                c = c.masked_fill(attention_mask[:, None, :] == 0, -1e4)
                pooled.append(c.max(2).values)
            z = torch.cat(pooled, 1)
        z = self.proj(self.drop(z))
        z = self.drop(z)
        out = {"logits": self.out(z)}
        if self.feat is not None and feats is not None:
            out["logits"] = out["logits"] + self.feat(feats)
        if self.aux2 is not None:
            out["aux2"], out["aux3"] = self.aux2(z), self.aux3(z)
        return out


def p_yes_from_logits(logits, target):
    probs = torch.softmax(logits.float(), dim=-1)
    if N_OUT[target] == 2:
        return probs[:, 1]
    if target == "cls4":
        return 1 - probs[:, 0]
    return (probs * torch.arange(7, device=probs.device) / 6).sum(1)      # cls7


def soft_cross_entropy(logits, target_dist):
    return -(target_dist * F.log_softmax(logits.float(), dim=-1)).sum(-1).mean()


def compute_loss(out, idx, cfg):
    t = cfg["target"]
    logits = out["logits"]
    if t in SOFT_TARGETS:
        p = torch.tensor(SOFT_TARGETS[t][idx], dtype=torch.float, device=device)
        loss = soft_cross_entropy(logits, torch.stack([1 - p, p], 1))
    elif t == "hard":
        p = torch.tensor(Y_TR[idx], dtype=torch.float, device=device)
        loss = soft_cross_entropy(logits, torch.stack([1 - p, p], 1))
    elif t == "cls4":
        loss = soft_cross_entropy(logits, torch.tensor(T2_TR[idx], dtype=torch.float, device=device))
    else:
        loss = F.cross_entropy(logits.float(), torch.tensor(NYES_TR[idx], dtype=torch.long, device=device))
    if t == "multitask":
        loss = loss + cfg["aux_w"] * (
            soft_cross_entropy(out["aux2"], torch.tensor(T2_TR[idx], dtype=torch.float, device=device))
            + F.binary_cross_entropy_with_logits(out["aux3"].float(), torch.tensor(T3_TR[idx], dtype=torch.float, device=device)))
    return loss


AEDA_PUNCT = [".", ";", "?", ":", "!", ","]


def aeda(text, rng):
    words = text.split()
    if not words:
        return text
    n_ins = int(rng.integers(1, max(2, len(words) // 3 + 1)))
    pos = set(rng.choice(len(words), size=min(n_ins, len(words)), replace=False).tolist())
    out = []
    for i, w in enumerate(words):
        if i in pos:
            out.append(AEDA_PUNCT[int(rng.integers(len(AEDA_PUNCT)))])
        out.append(w)
    return " ".join(out)


def param_groups(model, cfg):
    n_layers = model.encoder.config.num_hidden_layers
    groups = {}
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        no_wd = p.ndim == 1 or "LayerNorm" in n or "layernorm" in n or n.endswith(".bias")
        if n.startswith("encoder."):
            m = re.search(r"\.layer\.(\d+)\.", n)
            depth = 0 if "embeddings" in n else (int(m.group(1)) + 1 if m else n_layers)
            lr = cfg["lr"] * cfg["llrd"] ** (n_layers - depth)
        elif n.startswith("feat."):
            lr = 1e-3
        else:
            lr = cfg["head_lr"]
        key = (lr, 0.0 if no_wd else 0.01)
        groups.setdefault(key, []).append(p)
    return [{"params": ps, "lr": lr, "weight_decay": wd} for (lr, wd), ps in groups.items()]


def encode(tok, texts, max_len):
    enc = tok(list(texts), truncation=True, max_length=max_len, padding=True, return_tensors="pt")
    return enc["input_ids"].to(device), enc["attention_mask"].to(device)


@torch.no_grad()
def predict(model, tok, texts, feats, cfg, bs=128):
    model.eval()
    order = np.argsort([len(t) for t in texts])
    out = np.zeros(len(texts))
    for s in range(0, len(texts), bs):
        idx = order[s:s + bs]
        ids, mask = encode(tok, texts[idx], cfg["max_len"])
        f = torch.tensor(feats[idx], device=device) if cfg["feats"] else None
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            o = model(ids, mask, f)
        out[idx] = p_yes_from_logits(o["logits"], cfg["target"]).cpu().numpy()
    return out


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

In [14]:
def cfg_key(cfg):
    keys = ["model", "prep", "target", "head", "aeda", "feats", "epochs", "lr", "head_lr", "llrd", "bs", "max_len", "seed", "aux_w"]
    return "_".join(str(cfg[k]).replace("/", "-") for k in keys)


def _train_fold_once(cfg, fold, attempt=0):
    """Entraîne un fold ; met en cache les prédictions (validation et dev) de chaque époque."""
    d = CACHE_DIR / "runs" / cfg_key(cfg)
    path = d / (f"fold{fold}.npz" if attempt == 0 else f"fold{fold}_retry{attempt}.npz")
    if path.exists():
        return dict(np.load(path))
    d.mkdir(parents=True, exist_ok=True)
    seed = cfg["seed"] * 100 + fold + 1000 * attempt
    set_seed(seed)
    rng = np.random.default_rng(seed)

    tok = get_tokenizer(cfg["model"])
    texts_tr, texts_dev = TEXTS[cfg["prep"]]
    va_idx = np.where(FOLDS == fold)[0]
    tr_idx = np.where(FOLDS != fold)[0]
    if cfg["target"] == "hard":
        tr_idx = tr_idx[~TIE_TR[tr_idx]]

    model = TweetClassifier(cfg["model"], cfg["head"], cfg["target"], n_feats=HFZ_TR.shape[1] if cfg["feats"] else 0).to(device)
    optimizer = torch.optim.AdamW(param_groups(model, cfg), fused=True)
    copies = 2 if cfg["aeda"] else 1
    steps_per_epoch = math.ceil(len(tr_idx) * copies / cfg["bs"])
    total = steps_per_epoch * cfg["epochs"]
    scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total), total)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP_DTYPE == torch.float16)

    p_va, p_dev, losses = [], [], []
    for epoch in range(cfg["epochs"]):
        model.train()
        items = [(i, False) for i in tr_idx] + ([(i, True) for i in tr_idx] if cfg["aeda"] else [])
        items = [items[j] for j in rng.permutation(len(items))]
        ep_losses = []
        for s in tqdm(range(0, len(items), cfg["bs"]), desc=f"{cfg['name']} fold {fold} époque {epoch + 1}", leave=False):
            batch = items[s:s + cfg["bs"]]
            idx = np.array([i for i, _ in batch])
            texts = [aeda(texts_tr[i], rng) if aug else texts_tr[i] for i, aug in batch]
            ids, mask = encode(tok, texts, cfg["max_len"])
            f = torch.tensor(HFZ_TR[idx], device=device) if cfg["feats"] else None
            with torch.autocast("cuda", dtype=AMP_DTYPE):
                out = model(ids, mask, f)
            loss = compute_loss(out, idx, cfg)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update(); scheduler.step()
            ep_losses.append(loss.item())
        losses.append(np.mean(ep_losses))
        p_va.append(predict(model, tok, texts_tr[va_idx], HFZ_TR[va_idx], cfg))
        p_dev.append(predict(model, tok, texts_dev, HFZ_DEV, cfg))

    res = dict(va_idx=va_idx, p_va=np.array(p_va), p_dev=np.array(p_dev), loss=np.array(losses))
    np.savez(path, **res)
    del model, optimizer, scheduler
    gc.collect(); torch.cuda.empty_cache()
    return res


def fold_f1(res):
    va = res["va_idx"]; m = EVAL_TR[va]
    return max(f1_at(Y_TR[va][m], p[m], t) for p in res["p_va"] for t in (0.3, 0.4, 0.5))


def train_fold(cfg, fold, max_attempts=3):
    """Garde-fou : un fold qui a divergé (F1 < COLLAPSE_F1 à toutes les époques) est relancé avec une autre seed."""
    for attempt in range(max_attempts):
        res = _train_fold_once(cfg, fold, attempt)
        f1 = fold_f1(res)
        if f1 >= COLLAPSE_F1:
            return res
        print(f"  ⚠ {cfg['name']} fold {fold} : F1 {f1:.3f} < {COLLAPSE_F1} (divergence), relance n° {attempt + 1}")
    return res


EPOCH_TABLES = {}


def collect_folds(fold_fn, n_epochs):
    """OOF (chaque tweet prédit par le modèle qui ne l'a pas vu) et dev (moyenne des 5 modèles)."""
    oof = np.full((n_epochs, N_TR), np.nan)
    dev = np.zeros((n_epochs, N_DEV))
    for fold in range(N_FOLDS):
        r = fold_fn(fold)
        oof[:, r["va_idx"]] = r["p_va"]
        dev += r["p_dev"] / N_FOLDS
    return oof, dev


def select_epoch(name, oof, dev, kind, **info):
    """Choisit l'époque sur l'OOF F1 (seuil optimal), à égalité sur la cross-entropy OOF, et enregistre."""
    rows = []
    for e in range(oof.shape[0]):
        thr, f1b = best_threshold(oof[e])
        m = EVAL_TR & ~np.isnan(oof[e])
        rows.append({"époque": e + 1, "OOF F1@0.5": f1_at(Y_TR[m], oof[e][m]), "OOF F1@seuil": f1b,
                     "seuil": thr, "OOF CE": soft_ce(P_TR, oof[e]), "dev F1@0.5": f1_at(Y_DEV[EVAL_DEV], dev[e][EVAL_DEV])})
    table = pd.DataFrame(rows)
    best = int(table.sort_values(["OOF F1@seuil", "OOF CE"], ascending=[False, True]).index[0])
    EPOCH_TABLES[name] = table
    register(name, oof[best], dev[best], kind, epoch=best + 1, **info)
    print(f"{name:22s} époque retenue {best + 1} | OOF F1 {table.loc[best, 'OOF F1@seuil']:.4f} "
          f"| dev F1@0.5 {table.loc[best, 'dev F1@0.5']:.4f}")
    return table


def run_config(cfg, kind="transformer"):
    t0 = time.time()
    oof, dev = collect_folds(lambda fold: train_fold(cfg, fold), cfg["epochs"])
    select_epoch(cfg["name"], oof, dev, kind, cfg=cfg)
    print(f"   ({(time.time() - t0) / 60:.1f} min)")


# même BASE que modelling 2 après le choix du calendrier : les clés de cache coïncident
BASE = dict(model="cardiffnlp/twitter-xlm-roberta-large-2022", prep="light", target="soft", head="cls",
            aeda=False, feats=False, epochs=3, lr=1e-5, head_lr=1e-4, llrd=1.0, bs=16, max_len=128, seed=SEED, aux_w=0.5)


def make_cfg(name, **kw):
    c = dict(BASE, **kw)
    c["name"] = name
    return c


XLMR_POOL = []   # candidats de l'ensemble glouton


def run_seeds(prefix, kw, seeds):
    """Entraîne une configuration avec plusieurs seeds, enregistre chaque seed et la moyenne."""
    t0 = time.time()
    oofs, devs = [], []
    for s in seeds:
        cfg = make_cfg(f"{prefix}_s{s}", **kw, seed=s)
        oof, dev = collect_folds(lambda fold, cfg=cfg: train_fold(cfg, fold), cfg["epochs"])
        select_epoch(cfg["name"], oof, dev, "transformer", cfg=cfg)
        XLMR_POOL.append(cfg["name"])
        oofs.append(oof); devs.append(dev)
    select_epoch(prefix, np.mean(oofs, 0), np.mean(devs, 0), "transformer", seeds=list(seeds))
    XLMR_POOL.append(prefix)
    print(f"   {prefix} : {(time.time() - t0) / 60:.1f} min")

In [15]:
if RUN_XLMR:
    # référence (cache modelling 2)
    for t in ["soft", "hard"]:
        run_config(make_cfg(f"base_{t}", target=t))
    register("baseline", (REG["base_soft"]["oof"] + REG["base_hard"]["oof"]) / 2,
             (REG["base_soft"]["dev"] + REG["base_hard"]["dev"]) / 2, "transformer")

    SOFT_BEST = dict(epochs=3, lr=2e-5)                                   # meilleur calendrier soft de modelling 2
    run_seeds("xlmr_light", SOFT_BEST, XLMR_SEEDS)                         # cache modelling 2 (soft_s42 … s46)
    run_seeds("xlmr_v2_mt", dict(SOFT_BEST, prep="v2", target="multitask"), XLMR_SEEDS)
    run_seeds("xlmr_v2_cls4", dict(SOFT_BEST, prep="v2", target="cls4"), XLMR_SEEDS[:3])

    display(summary(["baseline", "base_soft"] + [n for n in XLMR_POOL if "_s" not in n]))

base_soft              époque retenue 3 | OOF F1 0.8506 | dev F1@0.5 0.8776
   (0.0 min)
base_hard              époque retenue 3 | OOF F1 0.8455 | dev F1@0.5 0.8635
   (0.0 min)
xlmr_light_s42         époque retenue 3 | OOF F1 0.8512 | dev F1@0.5 0.8833
xlmr_light_s43         époque retenue 3 | OOF F1 0.8533 | dev F1@0.5 0.8699
xlmr_light_s44         époque retenue 3 | OOF F1 0.8467 | dev F1@0.5 0.8758
xlmr_light_s45         époque retenue 3 | OOF F1 0.8522 | dev F1@0.5 0.8735
xlmr_light_s46         époque retenue 3 | OOF F1 0.8457 | dev F1@0.5 0.8739
xlmr_light             époque retenue 3 | OOF F1 0.8564 | dev F1@0.5 0.8786
   xlmr_light : 0.0 min
xlmr_v2_mt_s42         époque retenue 3 | OOF F1 0.8509 | dev F1@0.5 0.8775
xlmr_v2_mt_s43         époque retenue 3 | OOF F1 0.8549 | dev F1@0.5 0.8765
xlmr_v2_mt_s44         époque retenue 3 | OOF F1 0.8520 | dev F1@0.5 0.8657
xlmr_v2_mt_s45         époque retenue 3 | OOF F1 0.8541 | dev F1@0.5 0.8725
xlmr_v2_mt_s46         époque retenue 

,type,OOF F1@0.5,seuil OOF,OOF F1@seuil,OOF CE soft,dev F1@0.5,dev F1@seuil,IC95 bas,IC95 haut,dev EN,dev ES,dev CE soft,Δ dev vs réf,p(Δ≤0)
nom,,,,,,,,,,,,,,
xlmr_v2_mt,transformer,0.8561,0.48,0.8572,0.5231,0.8749,0.8770,0.8542,0.8984,0.8376,0.9067,0.4995,0.0092,0.1065
xlmr_v2_cls4,transformer,0.8504,0.43,0.8570,0.5261,0.8750,0.8791,0.8555,0.9008,0.8429,0.9064,0.5028,0.0113,0.0620
xlmr_light,transformer,0.8534,0.46,0.8564,0.5284,0.8786,0.8778,0.8555,0.8995,0.8471,0.9011,0.5027,0.0100,0.0645
base_soft,transformer,0.8506,0.50,0.8506,0.5369,0.8776,0.8776,0.8543,0.8991,0.8454,0.9017,0.5027,0.0098,0.1075
baseline,transformer,0.8450,0.27,0.8505,0.6054,0.8705,0.8678,0.8444,0.8896,0.8592,0.8743,0.5541,NaN,NaN


## 4. LLM 14B fine-tunés en LoRA bf16

Même principe que modelling 2 (P(YES) lue au premier token de la réponse, softmax sur les logits de « NO » et « YES », cross-entropy avec la cible soft, mêmes 5 folds), avec les leviers que les courbes de modelling 2 désignaient :

- **14B au lieu de 7B, en bf16 au lieu de 4 bits** : ~30 Go de poids, la G4 en a 96 ;
- **rang 32, α 64** sur toutes les projections (attention + MLP) ;
- **3 époques**, calendrier **cosinus**, warmup 5 %, LR 1e-4, lot effectif 16 ; l'époque est choisie sur l'OOF ;
- **consigne** : la définition EXIST en trois lignes (le fine-tuning sur une seule phrase obligeait le modèle à tout réapprendre) ;
- **deux familles** (Qwen2.5 et Qwen3, pré-entraînements différents) pour que l'ensemble profite d'erreurs décorrélées ; le 7B 4 bits de modelling 2 est relu depuis le cache et reste candidat.

Le tweet est tronqué à 600 caractères **avant** le gabarit de chat (jamais de troncature en tokens, qui couperait la question finale).

In [16]:
def vram(msg=""):
    print(f"{msg} VRAM allouée : {torch.cuda.memory_allocated(0) / 1e9:.2f} Go | réservée : {torch.cuda.memory_reserved(0) / 1e9:.2f} Go")


def load_tokenizer(name):
    tok = AutoTokenizer.from_pretrained(name)
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    return tok


def load_llm(name):
    tok = load_tokenizer(name)
    model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.bfloat16, device_map="cuda",
                                                 attn_implementation="sdpa")
    vram(f"{name} (bf16) chargé —")
    return tok, model


def chat_text(tok, prompt):
    msgs = [{"role": "user", "content": prompt}]
    try:
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)


def left_pad(tok, seqs):
    L = max(len(e) for e in seqs)
    ids = torch.full((len(seqs), L), tok.pad_token_id, dtype=torch.long)
    mask = torch.zeros((len(seqs), L), dtype=torch.long)
    for r, e in enumerate(seqs):
        ids[r, L - len(e):] = torch.tensor(e); mask[r, L - len(e):] = 1
    pos = (mask.cumsum(1) - 1).clamp(min=0)
    return ids.to(device), mask.to(device), pos.to(device)


def last_token_logits(causal_lm, ids, mask, pos):
    h = causal_lm.get_decoder()(input_ids=ids, attention_mask=mask, position_ids=pos, use_cache=False).last_hidden_state[:, -1]
    return causal_lm.lm_head(h).float()


def adaptive_batches(enc, order, token_budget, step, desc=None):
    """Lots d'environ `token_budget` tokens ; en cas d'OOM, le lot est divisé par deux et le budget réduit est conservé."""
    pbar = tqdm(total=len(order), desc=desc, disable=desc is None, leave=False)
    i = 0
    while i < len(order):
        L = len(enc[order[i]])
        n = max(1, token_budget // L)
        idx = order[i:i + n]
        oom = False
        try:
            step(idx)
        except torch.cuda.OutOfMemoryError:
            if n == 1:
                raise
            oom = True
        if oom:
            gc.collect(); torch.cuda.empty_cache()
            token_budget = max(L, (n // 2) * L)
            print(f"  OOM : lot de {n} → {max(1, token_budget // L)}")
            continue
        i += len(idx); pbar.update(len(idx))
    pbar.close()


FT_PROMPT_VERSION = "v3"
FT_INSTRUCTION = (
    "EXIST 2024 guidelines. A tweet is sexist (YES) if:\n"
    "- it is sexist itself: it insults, belittles, objectifies, stereotypes or threatens women because they are women, "
    "defends traditional gender roles, attacks feminism or denies women's rights (irony and sarcasm included);\n"
    "- it reports a sexist situation suffered by a woman or women (harassment, abuse, discrimination, gender violence), "
    "including personal experiences and news;\n"
    "- it describes or criticises sexist behaviour (machismo, wage gap, mansplaining, harassment...).\n"
    "It is not sexist (NO) if it is not about sexism: a topic word used in another sense, vulgarity or insults "
    "that do not target women as women, neutral or positive mentions of women, advertising, unrelated content.\n\n"
    "Tweet: {tweet}\n\nIs this tweet sexist? Answer YES or NO.")

LLM_TEXTS_TR, LLM_TEXTS_DEV = TEXTS["xlmt"]
print(FT_INSTRUCTION.format(tweet=LLM_TEXTS_DEV[0][:600]))

EXIST 2024 guidelines. A tweet is sexist (YES) if:
- it is sexist itself: it insults, belittles, objectifies, stereotypes or threatens women because they are women, defends traditional gender roles, attacks feminism or denies women's rights (irony and sarcasm included);
- it reports a sexist situation suffered by a woman or women (harassment, abuse, discrimination, gender violence), including personal experiences and news;
- it describes or criticises sexist behaviour (machismo, wage gap, mansplaining, harassment...).
It is not sexist (NO) if it is not about sexism: a topic word used in another sense, vulgarity or insults that do not target women as women, neutral or positive mentions of women, advertising, unrelated content.

Tweet: @user La comunidad gamer es un antro de misóginos y supremacistas blancos hace años pero hoy día pocos se acuerdan de gamergate y todo lo que causó, y en cierto sentido sigue causando.

Is this tweet sexist? Answer YES or NO.


In [17]:
import sys, gc, torch, peft
for _m in list(sys.modules.values()):
    if getattr(_m, "__name__", "").startswith("peft") and hasattr(_m, "is_torchao_available"):
        _m.is_torchao_available = lambda: False

# le 14B chargé avant l'erreur est encore en mémoire (~30 Go) : on le libère pour repartir propre
base = None
gc.collect(); torch.cuda.empty_cache()

In [18]:
def llm_tag(run):
    return (f"llmft_{run['model'].replace('/', '-')}_bf16_{run['epochs']}ep_lr{run['lr']}_r{run['r']}a{run['alpha']}"
            f"_bs{run['bs']}_s{run['seed']}_{FT_PROMPT_VERSION}")


def ft_fold(run, fold, tok, base, enc_tr, enc_dev, yes_id, no_id):
    path = CACHE_DIR / "runs" / llm_tag(run) / f"fold{fold}.npz"
    if path.exists():
        return dict(np.load(path))
    path.parent.mkdir(parents=True, exist_ok=True)
    from peft import LoraConfig, get_peft_model
    set_seed(run["seed"] * 100 + fold)
    rng = np.random.default_rng(run["seed"] * 100 + fold)
    lora = LoraConfig(r=run["r"], lora_alpha=run["alpha"], lora_dropout=run["dropout"], task_type="CAUSAL_LM",
                      target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
    model = get_peft_model(base, lora)
    if fold == 0:
        model.print_trainable_parameters()
    causal = model.get_base_model()
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=run["lr"], weight_decay=0.0, fused=True)
    tr_idx, va_idx = np.where(FOLDS != fold)[0], np.where(FOLDS == fold)[0]
    steps = math.ceil(len(tr_idx) / run["bs"]) * run["epochs"]
    scheduler = get_cosine_schedule_with_warmup(optimizer, int(run["warmup"] * steps), steps)

    @torch.no_grad()
    def predict_llm(encs):
        model.eval()
        order = np.argsort([-len(e) for e in encs]); out = np.zeros(len(encs))

        def step(idx):
            with torch.autocast("cuda", dtype=torch.bfloat16):
                lg = last_token_logits(causal, *left_pad(tok, [encs[j] for j in idx]))[:, [no_id, yes_id]]
            out[idx] = torch.softmax(lg, 1)[:, 1].cpu().numpy()

        adaptive_batches(encs, order, LLM_TOKEN_BUDGET, step)
        return out

    p_va, p_dev, losses = [], [], []
    t0 = time.time()
    for epoch in range(run["epochs"]):
        model.train()
        order = rng.permutation(tr_idx)
        ep = []
        for s in tqdm(range(0, len(order), run["bs"]), desc=f"{run['name']} fold {fold} époque {epoch + 1}", leave=False):
            idx = order[s:s + run["bs"]]
            optimizer.zero_grad(set_to_none=True)
            batch_loss = 0.0
            for m0 in range(0, len(idx), run["micro_bs"]):          # accumulation de gradient
                mi = idx[m0:m0 + run["micro_bs"]]
                with torch.autocast("cuda", dtype=torch.bfloat16):
                    lg = last_token_logits(causal, *left_pad(tok, [enc_tr[j] for j in mi]))[:, [no_id, yes_id]]
                p = torch.tensor(P_TR[mi], dtype=torch.float, device=device)
                loss = soft_cross_entropy(lg, torch.stack([1 - p, p], 1)) * len(mi) / len(idx)
                loss.backward()
                batch_loss += loss.item()
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            optimizer.step(); scheduler.step()
            ep.append(batch_loss)
        losses.append(np.mean(ep))
        p_va.append(predict_llm([enc_tr[j] for j in va_idx]))
        p_dev.append(predict_llm(enc_dev))
        m = EVAL_TR[va_idx]
        print(f"  {run['name']} fold {fold} époque {epoch + 1} : perte {losses[-1]:.4f}, "
              f"F1 du fold {best_f1_fold(Y_TR[va_idx][m], p_va[-1][m]):.4f} ({(time.time() - t0) / 60:.0f} min)")
    res = dict(va_idx=va_idx, p_va=np.array(p_va), p_dev=np.array(p_dev), loss=np.array(losses))
    np.savez(path, **res)
    model.unload()                       # retire les adaptateurs : le LLM de base sert au fold suivant
    model = optimizer = scheduler = None
    gc.collect(); torch.cuda.empty_cache()
    return res


def best_f1_fold(y, p):
    return max(f1_at(y, p, t) for t in THR_GRID)


LLM_POOL = []

# --- déjà en cache (modelling 2) : Qwen2.5-7B 4 bits et few-shot ---
OLD_FT = CACHE_DIR / "runs" / "llm_ft_Qwen-Qwen2.5-7B-Instruct_4bit_2ep_lr0.0001_r16"
if all((OLD_FT / f"fold{k}.npz").exists() for k in range(N_FOLDS)):
    oof, dev = collect_folds(lambda k: dict(np.load(OLD_FT / f"fold{k}.npz")), 2)
    select_epoch("llm_qwen25_7b_4bit", oof, dev, "LLM fine-tuné", model="Qwen/Qwen2.5-7B-Instruct")
    LLM_POOL.append("llm_qwen25_7b_4bit")
fs = CACHE_DIR / "llm_few_shot_k8_Qwen-Qwen2.5-7B-Instruct.npz"
if fs.exists():
    z = np.load(fs); register("llm_few_shot", z["tr"], z["dev"], "LLM")

# --- nouveaux LLM ---
if RUN_LLM_FT:
    for run in LLM_RUNS:
        t0 = time.time()
        done = all((CACHE_DIR / "runs" / llm_tag(run) / f"fold{k}.npz").exists() for k in range(N_FOLDS))
        base = None
        if done:
            tok = load_tokenizer(run["model"])
        else:
            gc.collect(); torch.cuda.empty_cache()
            tok, base = load_llm(run["model"])
            base.config.use_cache = False
            base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
            base.enable_input_require_grads()
        yes_id = tok.encode("YES", add_special_tokens=False)[0]
        no_id = tok.encode("NO", add_special_tokens=False)[0]
        assert yes_id != no_id, (yes_id, no_id)
        enc = lambda texts: [tok(chat_text(tok, FT_INSTRUCTION.format(tweet=t[:600])), add_special_tokens=False)["input_ids"]
                             for t in texts]
        ENC_TR, ENC_DEV = enc(LLM_TEXTS_TR), enc(LLM_TEXTS_DEV)
        print(f"{run['name']} : longueur moyenne {np.mean([len(e) for e in ENC_TR]):.0f} tokens, max {max(len(e) for e in ENC_TR)}")
        oof, dev = collect_folds(lambda k: ft_fold(run, k, tok, base, ENC_TR, ENC_DEV, yes_id, no_id), run["epochs"])
        select_epoch(run["name"], oof, dev, "LLM fine-tuné", model=run["model"])
        LLM_POOL.append(run["name"])
        display(EPOCH_TABLES[run["name"]].round(4))
        del base; gc.collect(); torch.cuda.empty_cache()
        print(f"   ({(time.time() - t0) / 60:.1f} min)")

display(summary([n for n in ["baseline", "xlmr_light", "xlmr_v2_mt"] + LLM_POOL if n in REG]))

llm_qwen25_7b_4bit     époque retenue 2 | OOF F1 0.8613 | dev F1@0.5 0.8705


Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

Qwen/Qwen2.5-14B-Instruct (bf16) chargé — VRAM allouée : 29.54 Go | réservée : 29.54 Go
llm_qwen25_14b : longueur moyenne 246 tokens, max 378


llm_qwen25_14b fold 1 époque 1:   0%|          | 0/346 [00:00<?, ?it/s]

  llm_qwen25_14b fold 1 époque 1 : perte 0.6752, F1 du fold 0.8590 (12 min)


llm_qwen25_14b fold 1 époque 2:   0%|          | 0/346 [00:00<?, ?it/s]

... (54 outputs left out)

  llm_qwen3_14b fold 4 époque 2 : perte 0.4707, F1 du fold 0.8556 (22 min)
llm_qwen3_14b          époque retenue 2 | OOF F1 0.8631 | dev F1@0.5 0.8818


,époque,OOF F1@0.5,OOF F1@seuil,seuil,OOF CE,dev F1@0.5
0,1,0.8582,0.8584,0.47,0.5101,0.8839
1,2,0.8587,0.8631,0.46,0.5046,0.8818


   (110.1 min)


,type,OOF F1@0.5,seuil OOF,OOF F1@seuil,OOF CE soft,dev F1@0.5,dev F1@seuil,IC95 bas,IC95 haut,dev EN,dev ES,dev CE soft,Δ dev vs réf,p(Δ≤0)
nom,,,,,,,,,,,,,,
llm_qwen3_14b,LLM fine-tuné,0.8587,0.46,0.8631,0.5046,0.8818,0.8923,0.8699,0.9129,0.8727,0.9070,0.4823,0.0245,0.0055
llm_qwen25_7b_4bit,LLM fine-tuné,0.8613,0.47,0.8613,0.5112,0.8705,0.8772,0.8524,0.8968,0.8571,0.8923,0.4887,0.0093,0.1905
llm_qwen25_14b,LLM fine-tuné,0.8573,0.47,0.8573,0.5095,0.8693,0.8792,0.8549,0.9011,0.8639,0.8906,0.4821,0.0113,0.1645
xlmr_v2_mt,transformer,0.8561,0.48,0.8572,0.5231,0.8749,0.8770,0.8542,0.8984,0.8376,0.9067,0.4995,0.0092,0.1065
xlmr_light,transformer,0.8534,0.46,0.8564,0.5284,0.8786,0.8778,0.8555,0.8995,0.8471,0.9011,0.5027,0.0100,0.0645
baseline,transformer,0.8450,0.27,0.8505,0.6054,0.8705,0.8678,0.8444,0.8896,0.8592,0.8743,0.5541,NaN,NaN


## 5. XLM-R XL (optionnel)

`facebook/xlm-roberta-xl` (3,5 B, non adapté à Twitter), fine-tuning complet en soft, `prep_v2`, LR 1e-5. Seul il ne devrait pas battre le LLM 14B ; il est là pour la diversité de l'ensemble. Le garde-fou de divergence est particulièrement utile ici.

In [19]:
XL_POOL = []
if RUN_XL:
    gc.collect(); torch.cuda.empty_cache()
    run_config(make_cfg("xl_v2", model="facebook/xlm-roberta-xl", prep="v2", epochs=3, lr=1e-5))
    XL_POOL.append("xl_v2")
    display(summary([n for n in ["baseline", "xlmr_v2_mt", "xl_v2"] if n in REG]))

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/405 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 13.9GB            

model.safetensors: downloading bytes:           |  0.00B            

... (18 outputs left out)

xl_v2 fold 4 époque 2:   0%|          | 0/346 [00:00<?, ?it/s]

xl_v2 fold 4 époque 3:   0%|          | 0/346 [00:00<?, ?it/s]

xl_v2                  époque retenue 3 | OOF F1 0.8491 | dev F1@0.5 0.8641
   (20.2 min)


,type,OOF F1@0.5,seuil OOF,OOF F1@seuil,OOF CE soft,dev F1@0.5,dev F1@seuil,IC95 bas,IC95 haut,dev EN,dev ES,dev CE soft,Δ dev vs réf,p(Δ≤0)
nom,,,,,,,,,,,,,,
xlmr_v2_mt,transformer,0.8561,0.48,0.8572,0.5231,0.8749,0.8770,0.8542,0.8984,0.8376,0.9067,0.4995,0.0092,0.1065
baseline,transformer,0.8450,0.27,0.8505,0.6054,0.8705,0.8678,0.8444,0.8896,0.8592,0.8743,0.5541,NaN,NaN
xl_v2,transformer,0.8449,0.44,0.8491,0.5216,0.8641,0.8730,0.8489,0.8951,0.8550,0.8864,0.5020,0.0052,0.2680


## 6. Combinaisons

Tout est appris **sur l'OOF** et appliqué tel quel au dev.

1. `xlmr_ens` : ensemble glouton (Caruana) sur tous les XLM-R (seeds et moyennes).
2. `llm_ens` : ensemble glouton sur les LLM fine-tunés.
3. `mix_llm_xlmr` : `w · llm_ens + (1 − w) · xlmr_ens`, w choisi sur l'OOF (modelling 2 : un tiers de LLM ; avec des LLM plus forts, w devrait monter).
4. `greedy_tout` : ensemble glouton sur tous les modèles (XLM-R, LLM, XL).
5. `stack_final` : régression logistique (CV imbriquée) sur les logits de `xlmr_ens`, de chaque LLM, de XL, du few-shot et du modèle 1, + langue ; `stack_final+features` ajoute les features manuelles.

Le **modèle retenu** est le meilleur selon l'OOF. Le seuil par langue n'est gardé que s'il gagne au moins 0,3 point d'OOF.

In [20]:
def greedy_ensemble(names, n_iter=20):
    chosen, best = [], -1
    for _ in range(n_iter):
        scores = {n: best_threshold(np.mean([REG[c]["oof"] for c in chosen + [n]], 0))[1] for n in names}
        n_best = max(scores, key=scores.get)
        if scores[n_best] <= best + 1e-5:
            break
        chosen.append(n_best); best = scores[n_best]
    return chosen


def register_mean(name, members, **info):
    register(name, np.mean([REG[n]["oof"] for n in members], 0), np.mean([REG[n]["dev"] for n in members], 0),
             "combinaison", inputs=members, **info)
    print(f"{name:20s} :", dict(Counter(members)))


COMBOS = []
pool_x = [n for n in XLMR_POOL if n in REG]
pool_l = [n for n in LLM_POOL if n in REG]
pool_xl = [n for n in XL_POOL if n in REG]

if pool_x:
    register_mean("xlmr_ens", greedy_ensemble(pool_x)); COMBOS.append("xlmr_ens")
if pool_l:
    register_mean("llm_ens", greedy_ensemble(pool_l)); COMBOS.append("llm_ens")
if pool_x and pool_l:
    rows = []
    for w in np.round(np.arange(0, 1.001, 0.05), 2):
        p = w * REG["llm_ens"]["oof"] + (1 - w) * REG["xlmr_ens"]["oof"]
        rows.append({"w LLM": w, "OOF F1@seuil": best_threshold(p)[1]})
    mix = pd.DataFrame(rows)
    w = float(mix.loc[mix["OOF F1@seuil"].idxmax(), "w LLM"])
    register("mix_llm_xlmr", w * REG["llm_ens"]["oof"] + (1 - w) * REG["xlmr_ens"]["oof"],
             w * REG["llm_ens"]["dev"] + (1 - w) * REG["xlmr_ens"]["dev"], "combinaison", w=w)
    COMBOS.append("mix_llm_xlmr")
    print(f"Poids du LLM retenu sur l'OOF : {w}")
    display(mix.set_index("w LLM").T.round(4))
if pool_x or pool_l:
    register_mean("greedy_tout", greedy_ensemble(pool_x + pool_l + pool_xl)); COMBOS.append("greedy_tout")

STACK_INPUTS = (["xlmr_ens"] if pool_x else []) + pool_l + pool_xl + \
               [n for n in ["llm_few_shot"] + MODEL1_INPUTS if n in REG]
if len(STACK_INPUTS) >= 2:
    fit_stack("stack_final", STACK_INPUTS)
    fit_stack("stack_final+features", STACK_INPUTS, with_hand=True)
    COMBOS += ["stack_final", "stack_final+features"]
    if pool_x and pool_l:
        fit_stack("stack_llm_xlmr", ["xlmr_ens"] + pool_l); COMBOS.append("stack_llm_xlmr")

display(summary(COMBOS + pool_l + pool_xl + [n for n in ["xlmr_light", "xlmr_v2_mt", "xlmr_v2_cls4", "baseline"] if n in REG]))

xlmr_ens             : {'xlmr_v2_mt': 2, 'xlmr_v2_mt_s45': 3, 'xlmr_light_s43': 1, 'xlmr_light_s42': 1}
llm_ens              : {'llm_qwen3_14b': 1, 'llm_qwen25_7b_4bit': 1}
Poids du LLM retenu sur l'OOF : 0.9


w LLM,0.00,0.05,0.10,0.15,0.20,0.25,0.30,0.35,0.40,0.45,...,0.55,0.60,0.65,0.70,0.75,0.80,0.85,0.90,0.95,1.00
OOF F1@seuil,0.8619,0.8625,0.8633,0.8647,0.866,0.8668,0.8699,0.8691,0.869,0.87,...,0.8707,0.8699,0.8704,0.8708,0.8706,0.8708,0.8709,0.8715,0.8708,0.8699


greedy_tout          : {'llm_qwen3_14b': 1, 'llm_qwen25_7b_4bit': 1, 'xlmr_light_s43': 1}


,type,OOF F1@0.5,seuil OOF,OOF F1@seuil,OOF CE soft,dev F1@0.5,dev F1@seuil,IC95 bas,IC95 haut,dev EN,dev ES,dev CE soft,Δ dev vs réf,p(Δ≤0)
nom,,,,,,,,,,,,,,
greedy_tout,combinaison,0.8688,0.47,0.8721,0.4997,0.8809,0.8957,0.8735,0.9148,0.8766,0.9101,0.4805,0.0278,0.0000
stack_llm_xlmr,combinaison,0.8693,0.45,0.8720,0.6084,0.8824,0.8948,0.8725,0.9151,0.8808,0.9052,0.5745,0.0269,0.0010
mix_llm_xlmr,combinaison,0.8691,0.49,0.8715,0.4996,0.8857,0.8874,0.8632,0.9075,0.8825,0.8911,0.4799,0.0196,0.0260
stack_final,combinaison,0.8697,0.47,0.8710,0.6098,0.8874,0.8939,0.8711,0.9135,0.8743,0.9084,0.5772,0.0260,0.0040
stack_final+features,combinaison,0.8685,0.47,0.8703,0.6143,0.8886,0.8941,0.8710,0.9145,0.8714,0.9109,0.5792,0.0262,0.0030
llm_ens,combinaison,0.8681,0.49,0.8699,0.5014,0.8841,0.8906,0.8670,0.9106,0.8906,0.8906,0.4812,0.0228,0.0125
llm_qwen3_14b,LLM fine-tuné,0.8587,0.46,0.8631,0.5046,0.8818,0.8923,0.8699,0.9129,0.8727,0.9070,0.4823,0.0245,0.0055
xlmr_ens,combinaison,0.8619,0.50,0.8619,0.5244,0.8776,0.8776,0.8540,0.8990,0.8483,0.8996,0.4990,0.0098,0.1120
llm_qwen25_7b_4bit,LLM fine-tuné,0.8613,0.47,0.8613,0.5112,0.8705,0.8772,0.8524,0.8968,0.8571,0.8923,0.4887,0.0093,0.1905


### 6.1 Seuil par langue (gardé seulement s'il gagne sur l'OOF)

In [21]:
def per_language(name):
    r = REG[name]
    thr, pred_oof, pred_dev = {}, np.zeros(N_TR), np.zeros(N_DEV)
    for lang in ["en", "es"]:
        m_tr = EVAL_TR & (LANG_TR == lang)
        scores = [f1_at(Y_TR[m_tr], r["oof"][m_tr], t) for t in THR_GRID]
        thr[lang] = float(THR_GRID[int(np.argmax(scores))])
        pred_oof[LANG_TR == lang] = r["oof"][LANG_TR == lang] >= thr[lang]
        pred_dev[LANG_DEV == lang] = r["dev"][LANG_DEV == lang] >= thr[lang]
    return thr, f1_at(Y_TR[EVAL_TR], pred_oof[EVAL_TR]), pred_dev


rows = []
for name in COMBOS:
    thr, f1_oof_lang, pred_dev = per_language(name)
    rows.append({"modèle": name, "seuil EN": thr["en"], "seuil ES": thr["es"],
                 "OOF F1 seuil unique": oof_score(name), "OOF F1 seuil par langue": f1_oof_lang,
                 "dev F1 seuil unique": evaluate(name)["dev F1@seuil"],
                 "dev F1 seuil par langue": f1_at(Y_DEV[EVAL_DEV], pred_dev[EVAL_DEV])})
LANG_TABLE = pd.DataFrame(rows).set_index("modèle").round(4)
LANG_TABLE

,seuil EN,seuil ES,OOF F1 seuil unique,OOF F1 seuil par langue,dev F1 seuil unique,dev F1 seuil par langue
modèle,,,,,,
xlmr_ens,0.50,0.50,0.8619,0.8619,0.8776,0.8776
llm_ens,0.45,0.49,0.8699,0.8705,0.8906,0.8874
mix_llm_xlmr,0.47,0.49,0.8715,0.8722,0.8874,0.8882
greedy_tout,0.45,0.48,0.8721,0.8733,0.8957,0.8945
stack_final,0.46,0.52,0.8710,0.8721,0.8939,0.8879
stack_final+features,0.46,0.47,0.8703,0.8707,0.8941,0.8941
stack_llm_xlmr,0.38,0.48,0.8720,0.8724,0.8948,0.8899


## 7. Tableau final et modèle retenu

Triés par OOF F1 au seuil optimal (critère de choix). Δ et p(Δ≤0) sont calculés contre la **baseline** ; la cellule suivante compare aussi au **meilleur modèle de modelling 2**.

In [22]:
final_table = summary()
final_table.to_csv(RESULTS_CSV)
print("Tableau enregistré dans", RESULTS_CSV)
final_table

Tableau enregistré dans /content/drive/MyDrive/TweetClassification/resultats_modelling3.csv


,type,OOF F1@0.5,seuil OOF,OOF F1@seuil,OOF CE soft,dev F1@0.5,dev F1@seuil,IC95 bas,IC95 haut,dev EN,dev ES,dev CE soft,Δ dev vs réf,p(Δ≤0)
nom,,,,,,,,,,,,,,
greedy_tout,combinaison,0.8688,0.47,0.8721,0.4997,0.8809,0.8957,0.8735,0.9148,0.8766,0.9101,0.4805,0.0278,0.0000
stack_llm_xlmr,combinaison,0.8693,0.45,0.8720,0.6084,0.8824,0.8948,0.8725,0.9151,0.8808,0.9052,0.5745,0.0269,0.0010
mix_llm_xlmr,combinaison,0.8691,0.49,0.8715,0.4996,0.8857,0.8874,0.8632,0.9075,0.8825,0.8911,0.4799,0.0196,0.0260
stack_final,combinaison,0.8697,0.47,0.8710,0.6098,0.8874,0.8939,0.8711,0.9135,0.8743,0.9084,0.5772,0.0260,0.0040
stack_final+features,combinaison,0.8685,0.47,0.8703,0.6143,0.8886,0.8941,0.8710,0.9145,0.8714,0.9109,0.5792,0.0262,0.0030
llm_ens,combinaison,0.8681,0.49,0.8699,0.5014,0.8841,0.8906,0.8670,0.9106,0.8906,0.8906,0.4812,0.0228,0.0125
llm_qwen3_14b,LLM fine-tuné,0.8587,0.46,0.8631,0.5046,0.8818,0.8923,0.8699,0.9129,0.8727,0.9070,0.4823,0.0245,0.0055
xlmr_ens,combinaison,0.8619,0.50,0.8619,0.5244,0.8776,0.8776,0.8540,0.8990,0.8483,0.8996,0.4990,0.0098,0.1120
llm_qwen25_7b_4bit,LLM fine-tuné,0.8613,0.47,0.8613,0.5112,0.8705,0.8772,0.8524,0.8968,0.8571,0.8923,0.4887,0.0093,0.1905


In [24]:
CANDIDATES = [n for n in REG if REG[n]["kind"] in ("transformer", "combinaison", "LLM fine-tuné")]
FINAL = max(CANDIDATES, key=oof_score)
thr_final, _ = best_threshold(REG[FINAL]["oof"])
pred_final = (REG[FINAL]["dev"] >= thr_final).astype(int)
use_lang = FINAL in LANG_TABLE.index and \
    LANG_TABLE.loc[FINAL, "OOF F1 seuil par langue"] >= LANG_TABLE.loc[FINAL, "OOF F1 seuil unique"] + 0.003
if use_lang:
    thr_lang, _, pred_final = per_language(FINAL)
    pred_final = pred_final.astype(int)
    print("Seuil par langue retenu :", thr_lang)
f1_final = f1_score(Y_DEV[EVAL_DEV], pred_final[EVAL_DEV])
print(f"Modèle retenu : {FINAL} | OOF F1 {oof_score(FINAL):.4f} | seuil {thr_final} | dev F1 = {f1_final:.4f}")

# contre la baseline
if REFERENCE in REG:
    thr_ref, _ = best_threshold(REG[REFERENCE]["oof"])
    d, p = compare_dev(pred_final[EVAL_DEV], (REG[REFERENCE]["dev"][EVAL_DEV] >= thr_ref).astype(int))
    print(f"vs baseline            : Δ = {d:+.4f} | p(Δ≤0) = {p:.3f}")

# contre les meilleurs modèles de modelling 2 (prédictions dev et seuils OOF relus dans les CSV)
if PREV_PRED_CSV.exists() and PREV_RES_CSV.exists():
    prev = pd.read_csv(PREV_PRED_CSV).set_index("id_EXIST").loc[dev_df["id"]]
    prev_res = pd.read_csv(PREV_RES_CSV, index_col=0)
    for name in ["stack_tout", "ens_glouton+llm_ft", "stack_tout+features"]:
        if f"p_{name}" in prev and name in prev_res.index:
            ref_pred = (prev[f"p_{name}"].to_numpy()[EVAL_DEV] >= prev_res.loc[name, "seuil OOF"]).astype(int)
            d, p = compare_dev(pred_final[EVAL_DEV], ref_pred)
            print(f"vs {name:20s}: Δ = {d:+.4f} | p(Δ≤0) = {p:.3f}")

from sklearn.metrics import classification_report
print(classification_report(Y_DEV[EVAL_DEV], pred_final[EVAL_DEV], target_names=["NO", "YES"], digits=3))
err = pd.DataFrame({"n_yes": dev_df["n_yes"], "lang": LANG_DEV, "ok": pred_final == Y_DEV})[EVAL_DEV]
display(err.groupby("n_yes")["ok"].agg(["count", "mean"]).rename(columns={"mean": "exactitude"}).round(3))
display(err.groupby("lang")["ok"].agg(["count", "mean"]).rename(columns={"mean": "exactitude"}).round(3))

Modèle retenu : greedy_tout | OOF F1 0.8721 | seuil 0.47 | dev F1 = 0.8957
vs baseline            : Δ = +0.0278 | p(Δ≤0) = 0.000
vs stack_tout          : Δ = +0.0132 | p(Δ≤0) = 0.013
vs ens_glouton+llm_ft  : Δ = +0.0075 | p(Δ≤0) = 0.045
vs stack_tout+features : Δ = +0.0065 | p(Δ≤0) = 0.118
              precision    recall  f1-score   support

          NO      0.908     0.889     0.899       479
         YES      0.886     0.905     0.896       455

    accuracy                          0.897       934
   macro avg      0.897     0.897     0.897       934
weighted avg      0.897     0.897     0.897       934



,count,exactitude
n_yes,,
0,203,0.995
1,160,0.888
2,116,0.707
4,144,0.799
5,148,0.926
6,163,0.982


,count,exactitude
lang,,
en,444,0.890
es,490,0.904


In [25]:
out = dev_df[["id_EXIST", "lang", "tweet", "n_yes", "p_yes", "hard"]].copy()
out["mot_cle"] = SEED_DEV
for name in REG:
    out[f"p_{name}"] = REG[name]["dev"]
out["pred"] = np.where(pred_final == 1, "YES", "NO")
out["error_type"] = np.select([EVAL_DEV & (pred_final == 1) & (Y_DEV == 0), EVAL_DEV & (pred_final == 0) & (Y_DEV == 1)], ["FP", "FN"], default="")
out.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"{len(out)} tweets enregistrés dans {OUTPUT_CSV} ; erreurs : {out['error_type'].value_counts().to_dict()}")

print("\nFaux négatifs les plus nets :")
display(out[out["error_type"] == "FN"].sort_values(f"p_{FINAL}")[["lang", "tweet", "n_yes", f"p_{FINAL}"]].head(15))
print("Faux positifs les plus nets :")
display(out[out["error_type"] == "FP"].sort_values(f"p_{FINAL}", ascending=False)[["lang", "tweet", "n_yes", f"p_{FINAL}"]].head(15))

1038 tweets enregistrés dans /content/drive/MyDrive/TweetClassification/dev_predictions_v3.csv ; erreurs : {'': 942, 'FP': 53, 'FN': 43}

Faux négatifs les plus nets :


,lang,tweet,n_yes,p_greedy_tout
35,es,"Absuelto, si, pero, quién le devuelve su vida?. Cuanto daño el pvto #YoSiTeCreo @IreneMontero @Adrilastra #ComunismoEsMiseria ...",4,0.124478
446,es,"El lobo Carrasco si puede le come la polla a Messi a dos manos, que cansino",4,0.176211
660,en,@Lucesheffield @JuliaPa28346879 What is 'difference feminism'?? Never heard of this before.,4,0.221233
112,es,con q motivación voy a entrenar ahora si ya no veo a ninguna de las chicas 😭,5,0.229589
614,en,Tankie Centricide:-HE SCREAMS FRENCH PERSON RAISED BY SLAVIC FAMILY-he just like me frfr-has may 68 posters in his room-bitches about AB...,5,0.233749
219,es,"👉🏼Corrección: La @UESoficial, pese que es una institución pública, no cuenta con Unidad de Género. La instancia a la que señalamos de in...",4,0.239013
802,en,wonwoo cross legs &amp; jun manspreading 🦋🦋🦋🦋 @pledis_17 https://t.co/2pFi9Lzeji,4,0.260187
149,es,"@Bautiicapitani Uno te pregunta preocupado, y el culo parado no saluda por la calle. Mala persona",4,0.264526
82,es,"#EnVivo Conferencia ""Barreras invisibles para madres trabajadoras"", impartida por la Mtra. Virginia Shaday Gutiérrez León https://t.co/a...",5,0.272491
860,en,"Cyborg politics is the struggle for language and the struggle against perfect communication, against the one code that translates all me...",5,0.276126


Faux positifs les plus nets :


,lang,tweet,n_yes,p_greedy_tout
973,en,@bhargavtata0 oh so now u can’t understand women’s words??????,2,0.814165
549,en,"@Mike_Fabricant “You should smile more, love. Just pretend you’re a Tory opening a food bank. We’re only living through the worst #CostO...",2,0.802604
785,en,@BoopBepp @Helloth07860784 @Bronwen85 Yup I hate when men rape and kill women. https://t.co/MyfVPu6THd,2,0.782412
1029,en,"Uh, they're allowed to? calling this out makes you look like a cunt. Fortunately, I'll block you so fast you wont get your screenshot. h...",2,0.765803
430,es,"La gente que come el chicle con la boca abierta? 🥴 Amiga, pareces una puta vaca pastando. Iiiiiiuuuuuu😷",2,0.756541
358,es,@clitfreaks @Andres54167596 Soy muy cortado aunque me gusta masturbarme viendo como se follan por el culo todavia soy virgen me gustaria...,1,0.731885
476,es,"Siempre y lo será una mierda, con su transfobia, machismo, homofobia, sexismo, xenofobia, y los personajes insoportables, y la gente tie...",2,0.728819
451,es,"@PSOE @Adrilastra La prostitución forzada, el proxenetismo, el rufianismo, la trata o como queráis llamarlo, ya está contemplado en el C...",2,0.727305
733,en,"“I mean I do, but… wouldn’t it be fun to get gangbanged together~?” https://t.co/lNRZ9z53VL",2,0.723387
53,es,"@Not_TsundereZ CalvoAh diablos no me denuncies por acoso sexual 😭😭Pero fuera bromas, que tiene que ver una cosa con la otra yo tampoco e...",2,0.720158


## 8. Lecture des résultats

| Question | Lignes à comparer |
|---|---|
| Le LLM 14B bf16 bat-il le 7B 4 bits ? | `llm_qwen25_14b`, `llm_qwen3_14b` vs `llm_qwen25_7b_4bit` (et tableau par époque : la 3e époque aide-t-elle ?) |
| Deux familles de LLM valent-elles mieux qu'une ? | `llm_ens` vs le meilleur LLM seul |
| `prep_v2` + multi-tâches cumulent-ils ? | `xlmr_v2_mt` vs `xlmr_light` |
| XL apporte-t-il de la diversité ? | `greedy_tout` avec / sans `xl_v2` dans sa composition |
| Quel poids pour le LLM ? | tableau `w LLM` de la partie 6 |
| Gain final | « Modèle retenu » vs baseline et vs `stack_tout` (modelling 2) |

**Si le gain du 14B est net**, l'étape suivante est le 32B (ligne commentée dans les paramètres), puis le réentraînement du modèle retenu sur **train + dev** pour la soumission sur le test.